In [ ]:
%run ./0___Reference.ipynb
%run ./0___Reference___Functions.ipynb
%run ./0___Reference___Plots.ipynb
%run ./0___Reference___Follow_Curve.ipynb
%run ./0___Reference___Shapes.ipynb
%run ./7___Finder___Functions.ipynb

In [ ]:
del all_parameters, file, ap, ALL_cores_no___1_NGP, ALL_cores_no___2_Smoothing, ALL_cores_no___3_Layer, ALL_cores_no___5_UOD_vals_and_lvls, ALL_cores_no___4_Origins, ALL_cores_no___6_Levels_isolated_pairs, ALL_cores_no___7_Finder, edge_cut, Density_uod, txt_ZONE, cdf_log, l_grid, i0, size, lvl, z, zPaths_no, sigma_ps, d_xyz_sgm; gc.collect()

In [ ]:
import plotly
import plotly.graph_objs as go
import plotly.offline as pyo
import matplotlib.patches as patches

from scipy.optimize import curve_fit
from scipy.stats import norm
from scipy.optimize import minimize

import matplotlib.colors as colors
from scipy.ndimage import gaussian_filter1d

from scipy.fft import  fftn as  fftn
from scipy.fft import ifftn as ifftn
from scipy.ndimage import fourier_gaussian as fourier_gaussian
from matplotlib.patches import Rectangle

---
---
---
---
---
---
---
---
---
---

In [ ]:
sz_indx = 2
Zs = ALL_Zs[sz_indx]

size = BASELINES[sz_indx]["size"]
ecc  = BASELINES[sz_indx]["ecc"]; d_xyz = (ecc[0][1]-ecc[0][0])/size
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
lvl  = BASELINES[sz_indx]["lvl"]
cl   = BASELINES[sz_indx]["cl"]
uod  = BASELINES[sz_indx]["uod"]; ud, od = uod
MK   = BASELINES[sz_indx]["MK"]

s_r = range(size)

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

---
---
---
---
---
---
---
---
---
---

# 11. Get all Ellipsoid Fits and Plot the Axis Radii Ratios

---
---
---

## 11.1 Fit Ellipsoids on all Voids

---

### For all Z's at preferred CDF range

In [ ]:
for Z in ALL_Zs[sz_indx]:
    file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
    if not os.path.exists(file_path_ellips_Z):                      os.makedirs(file_path_ellips_Z)
    if not os.path.exists(file_path_ellips_Z+"voids_edge_coords/"): os.makedirs(file_path_ellips_Z+"voids_edge_coords/")
    if not os.path.exists(file_path_ellips_Z+"all_connections/"):   os.makedirs(file_path_ellips_Z+"all_connections/")
    if not os.path.exists(file_path_ellips_Z+"all_directions/"):    os.makedirs(file_path_ellips_Z+"all_directions/")
    if not os.path.exists(file_path_ellips_Z+"all_directions/"):    os.makedirs(file_path_ellips_Z+"all_directions/")
    if not os.path.exists(file_path_ellips_Z+"patches_combined/"):  os.makedirs(file_path_ellips_Z+"patches_combined/")
    if not os.path.exists(file_path_ellips_Z+"void_combined/"):     os.makedirs(file_path_ellips_Z+"void_combined/")
    if not os.path.exists(file_path_ellips_Z+"Ellipsoid_values/"):  os.makedirs(file_path_ellips_Z+"Ellipsoid_values/")

---
---
---

In [ ]:
# If we look only at voids that do not interact with the mirrored elements (past the edge).
only_inside = False

# If we have a void which completely engulfs another, we may chose to keep it or disregrard it from our analysis.
keep_engulfing_voids = True

running = True
cores_no___Analysis___Ellipsoids_and_Fits_1 = 4

In [ ]:
notebook_filenames = []; notebook_parameters = []

for i0 in range(len(Zs)):
    
    Z = ALL_Zs[sz_indx][i0]
    file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
    
    file_path_Z    = file_path     +"Z___"  +Z                   +"/"
    file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
    file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
    file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
    file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
    file_path_MK   = file_path_Duod+MK                           +"/"
    
    notebook_filenames.append("Analysis___Ellipsoids_and_Fits_1.ipynb")
    notebook_parameters.append({"size":                 size,
                                "file_path_MK":         file_path_MK,
                                "only_inside":          only_inside,
                                "keep_engulfing_voids": keep_engulfing_voids,
                                "file_path_ellips_Z":   file_path_ellips_Z})

run_the_notebooks(running, notebook_filenames, notebook_parameters, notebook_directory, output_directory, cores_no___Analysis___Ellipsoids_and_Fits_1)

---
---
---

## 11.2 Color Histogram all Z's

---

In [ ]:
l_center = []; l_radii = []; l_evecs = []; l_v = []; l_i1 = []

for Z in ALL_Zs[sz_indx]:
    file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
    with open(file_path_ellips_Z+"Ellipsoid_values/l_center.pk",    'rb') as f: l_center.append(pkl.load(f))
    with open(file_path_ellips_Z+"Ellipsoid_values/l_radii.pk",     'rb') as f: l_radii.append( pkl.load(f))
    with open(file_path_ellips_Z+"Ellipsoid_values/l_evecs.pk",     'rb') as f: l_evecs.append( pkl.load(f))
    with open(file_path_ellips_Z+"Ellipsoid_values/l_v.pk",         'rb') as f: l_v.append(     pkl.load(f))
    with open(file_path_ellips_Z+"Ellipsoid_values/l_i1.pk",        'rb') as f: l_i1.append(    pkl.load(f))

In [ ]:
Zs    = ALL_Zs[         sz_indx]
Zs_f  = ALL_Zs_float[   sz_indx]
Zs_fs = ALL_Zs_floatstr[sz_indx]

---

In [ ]:
max_cells1 = 200   # max cells we display the longest radii (a)
max_cells2 = 137

no_bins = math.ceil(10/d_xyz)

In [ ]:
max_cells1 * d_xyz, max_cells2 * d_xyz

In [ ]:
pixels = []
max_pixels = []
rng_max  = [max_cells1, max_cells2, 1, 1]
r_mean = []

for i00 in range(len(l_radii)):
    
    pixels.append([])
    
    r1 = [i[0] for i in l_radii[i00]]
    r2 = [i[1] for i in l_radii[i00]]
    r3 = [i[2] for i in l_radii[i00]]
    
    # reorder r3>r2>r1
    for i in range(len(r1)):
        rr = sorted([r1[i], r2[i], r3[i]])
        r1[i] = rr[0]; r2[i] = rr[1]; r3[i] = rr[2]
    
    r123 = [np.array(r3), np.array(r2)/np.array(r3), np.array(r1)/np.array(r3)]

    r_mean.append([1., np.mean(r123[1]), np.mean(r123[2])])

    for i0 in range(4):
        i01 = 0
        if i0 >= 1: i01 = i0-1
        plt.figure(figsize=(14,6), dpi=100)
        counts, bins, _ = plt.hist(r123[i01], density=True, bins=no_bins, range=(0,rng_max[i0]))
        pixels[-1].append(counts)
        max_pixels.append(np.max(counts))
        plt.close()
r_mean = np.asarray(r_mean, dtype=float)

In [ ]:
pixels_copy = copy.deepcopy(pixels)

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(10,14), dpi=400, sharex=True)

fig.suptitle(  r"Distribution of the 3-axis conic radii for each Z" + "\n"
             +       "size="+str(size)
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")




for i00 in range(4):

    pixels = np.array([pixels_copy[i][i00] for i in range(len(pixels_copy))]).T[::-1]

    imm = axs[i00].imshow(pixels, cmap='inferno', extent=[0.0-0.5, len(Zs)-0.5, 0, rng_max[i00]], aspect='auto')

    divider   = make_axes_locatable(axs[i00])
    cax       = divider.append_axes('right', size='3%', pad=0.05)
    cbar      = plt.colorbar(imm, cax=cax, orientation='vertical')
    tks, tkss, stbl_exp = set_ticks(np.min(pixels), np.max(pixels), log_lin=False, int_if_possible=True, auto_stable_exponent=True)
    cbar.ax.set_yticks(tks, tkss)
    if stbl_exp != 0: cbar.ax.text(.5, 1.01, rf'$\times 10^{{{stbl_exp}}}$', ha='center', va='bottom', transform=cbar.ax.transAxes, fontsize=10)
    cbar.set_label('Distribution density (normed)', rotation=270, labelpad=5)
    
    
    
    for i0 in range(len(Zs)-1):
        axs[i00].axvline(x=i0+0.5, color='black', linestyle='-', linewidth=0.3)
    
    
axs[0].set_ylabel('A\n[cMpc/h]')
axs[1].set_ylabel('A\n[cMpc/h]')
axs[2].set_ylabel('B/A')
axs[3].set_ylabel('C/A')

plt.xlabel('Z')

axs[2].set_xticks([_ for _ in range(len(Zs))], Zs_fs)
for i00 in range(2):
    tks, tkss = set_ticks(0, rng_max[i00]*d_xyz, log_lin=False, int_if_possible=True, float_g=True, number_of_tks_clean_max=4)
    tks = [_/d_xyz for _ in tks]
    #tkss = [latex_float(_*d_xyz, float_g=True) for _ in tks]
    axs[i00].set_yticks(tks, tkss)


plt.tight_layout()
plt.savefig(plots_path_0+"11.2___hist_3_color_plot.png")
plt.close()

---
---
---

In [ ]:
if not os.path.exists(plots_path_0+"11.2___3D_grid_abc"): os.makedirs(plots_path_0+"11.2___3D_grid_abc")

---

In [ ]:
# rotation
ROT_X_DEG =   10
ROT_Y_DEG =    0
ROT_Z_DEG =  -25

# camera view
VIEW_ELEV = 18
VIEW_AZIM = -60
VIEW_ROLL = 0

# framing
MAX_SEMIAXIS = max(1.0, np.max(r_mean[:, 1:]))
BASE_FRAME_SCALE = 1.25
ZOOM_FACTOR = 2.7
FIXED_LIMIT = (BASE_FRAME_SCALE * MAX_SEMIAXIS) / ZOOM_FACTOR
DPI = 300

In [ ]:
# surface
SURFACE_COLOR = "#F4C56A"
SURFACE_ALPHA = 0.34

# wireframe
WIREFRAME_FRONT_COLOR = "#79CCFA"
WIREFRAME_BACK_COLOR  = "#447FA8"
WIREFRAME_FRONT_ALPHA = 0.98
WIREFRAME_BACK_ALPHA  = 0.88
WIREFRAME_FRONT_LINEWIDTH = 0.90
WIREFRAME_BACK_LINEWIDTH  = 0.72
WIREFRAME_RCOUNT = 18
WIREFRAME_CCOUNT = 28

# principal axes
A_COLOR = "#D64545"
B_COLOR = "#3AA65A"
C_COLOR = "#3366CC"
AXIS_LINEWIDTH = 2.2
AXIS_ALPHA = 1.0

# surface resolution
NU = 140
NV = 80

---

In [ ]:
FIGSIZE = (7,7)

In [ ]:
R_obj = rotation_matrix_xyz(ROT_X_DEG, ROT_Y_DEG, ROT_Z_DEG)
view_dir = get_view_direction(VIEW_ELEV, VIEW_AZIM)

for i, (a, b, c) in enumerate(r_mean):
    
    X,  Y,  Z  = make_ellipsoid_mesh_poles_along_A(a, b, c, nu=NU, nv=NV)
    Xr, Yr, Zr = rotate_mesh(X, Y, Z, R_obj)

    facecolors = make_surface_facecolors(X, Y, Z, a, b, c, R_obj, view_dir, base_color=SURFACE_COLOR, base_alpha=SURFACE_ALPHA)
    segments = collect_wireframe_segments(X, Y, Z, Xr, Yr, Zr, a, b, c, R_obj, view_dir, rcount=WIREFRAME_RCOUNT, ccount=WIREFRAME_CCOUNT)




    
    fig = plt.figure(figsize=FIGSIZE, dpi=250)
    ax = fig.add_subplot(111, projection="3d")
    ax.set_proj_type("ortho")

    try:              ax.computed_zorder = False
    except Exception: pass

    ax.set_position([0.00, 0.00, 1.00, 1.00])
    #ax.set_position([0.01, 0.01, 0.98, 0.98])
    ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM, roll=VIEW_ROLL)
    #ax.set_xlim(-FIXED_LIMIT, FIXED_LIMIT)
    #ax.set_ylim(-FIXED_LIMIT, FIXED_LIMIT)
    #ax.set_zlim(-FIXED_LIMIT, FIXED_LIMIT)
    #ax.set_box_aspect([1, 1, 1])
    fig_w, fig_h = FIGSIZE
    fig_ratio = fig_w / fig_h
    
    ax.set_xlim(-FIXED_LIMIT * fig_ratio, FIXED_LIMIT * fig_ratio)
    ax.set_ylim(-FIXED_LIMIT, FIXED_LIMIT)
    ax.set_zlim(-FIXED_LIMIT, FIXED_LIMIT)
    
    ax.set_box_aspect([fig_w, fig_h, fig_h])
    ax.set_axis_off()


    

    
    # surface
    ax.plot_surface(Xr, Yr, Zr, facecolors=facecolors, linewidth=0, antialiased=True, shade=False, zorder=1)

    # back-side wireframe
    draw_wireframe_subset(ax, segments, front_subset=False, color=WIREFRAME_BACK_COLOR, linewidth=WIREFRAME_BACK_LINEWIDTH, alpha=WIREFRAME_BACK_ALPHA, zorder_base=100)

    # 3) Internal axes
    A1 = np.array([-a,  0,  0])
    A2 = np.array([+a,  0,  0])
    B1 = np.array([ 0, -b,  0])
    B2 = np.array([ 0, +b,  0])
    C1 = np.array([ 0,  0, -c])
    C2 = np.array([ 0,  0, +c])

    A1r = R_obj @ A1
    A2r = R_obj @ A2
    B1r = R_obj @ B1
    B2r = R_obj @ B2
    C1r = R_obj @ C1
    C2r = R_obj @ C2

    draw_principal_axis(ax, A1r, A2r, A_COLOR, lw=AXIS_LINEWIDTH, alpha=AXIS_ALPHA, zorder=5000)
    draw_principal_axis(ax, B1r, B2r, B_COLOR, lw=AXIS_LINEWIDTH, alpha=AXIS_ALPHA, zorder=5001)
    draw_principal_axis(ax, C1r, C2r, C_COLOR, lw=AXIS_LINEWIDTH, alpha=AXIS_ALPHA, zorder=5002)

    # front-side wireframe
    draw_wireframe_subset(ax, segments, front_subset=True, color=WIREFRAME_FRONT_COLOR, linewidth=WIREFRAME_FRONT_LINEWIDTH, alpha=WIREFRAME_FRONT_ALPHA, zorder_base=10000)
    
    x = 0.015; y = 0.965
    fig.text(x, y, "z = " + Zs_fs[i], color="black", fontsize=14, ha="left", va="top")
    
    x = 0.985; y = 0.965; dy = 0.045
    fig.text(x, y - 0*dy, f"A   = {a:.3f}",   color=A_COLOR, fontsize=12, ha="right", va="top")
    fig.text(x, y - 1*dy, f"B/A = {b:.3f}",   color=B_COLOR, fontsize=12, ha="right", va="top")
    fig.text(x, y - 2*dy, f"C/A = {c:.3f}",   color=C_COLOR, fontsize=12, ha="right", va="top")


    
    plt.savefig(plots_path_0+"11.2___3D_grid_abc/ellipsoid___"+Zs[i]+".png", dpi=DPI, pad_inches=0.0)
    plt.close()

---

In [ ]:
INPUT_FOLDER = Path(plots_path_0+"11.2___3D_grid_abc/")

PREFIX    = None
EXTENSION = ".png"

OUTPUT_NAME   = "ellipsoid_video.mp4"
OUTPUT_FOLDER = None   # None = same as INPUT_FOLDER

DISPLAY_SECONDS = 0.5
VIDEO_FPS       = int(1/DISPLAY_SECONDS)   # I mean, you could always use sth else...

CODEC = "avc1"   # I can see this one with QuickTime Player but feel free to change it

make_video(reverse_ORDER = True)

---
---
---

## 11.3 3D Plot Void Coordinates & Fitted Ellipsoid

---

In [ ]:
if not os.path.exists(plots_path_0+"11.3___3D_grid"): os.makedirs(plots_path_0+"11.3___3D_grid")

---

In [ ]:
sz_indx = 2   # 512

size = BASELINES[sz_indx]["size"]
Z    = BASELINES[sz_indx]["Z"]
uod  = BASELINES[sz_indx]["uod"]; ud, od = uod

In [ ]:
file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
with open(file_path_ellips_Z+"Ellipsoid_values/l_center.pk", 'rb') as f: l_center = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_radii.pk",  'rb') as f: l_radii  = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_evecs.pk",  'rb') as f: l_evecs  = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_v.pk",      'rb') as f: l_v      = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_i1.pk",     'rb') as f: l_i1     = pkl.load(f)

In [ ]:
i1 = 247   # 5
#i1 = 367   # 4
i1 = 234   # 2 - but the ellipsoid is split in 4, that's why it's so cool

i11 = np.argwhere(np.array(l_i1) == i1)[0][0]

# Check if was able to be turned into an ellipsoid fit.
print("GOOD_ellipsoid:", i1 in l_i1)

In [ ]:
no_patches = 1
try:
    with open(file_path_ellips_Z+"all_connections/"+str(i1)+".pk", 'rb') as f: no_patches = len(pkl.load(f))
except: pass
print("no_patches =", no_patches)

In [ ]:
center = l_center[i11]; evecs = l_evecs[i11]; radii = l_radii[i1]; v = l_v[i11]

In [ ]:
with open(file_path_ellips_Z+"patches_combined/"+str(i1)+".pk", 'rb') as f: patches_combined = pkl.load(f)

---

In [ ]:
u = np.linspace(0.0, 2.0 * np.pi, 400); v = np.linspace(0.0, np.pi, 400)

xm, ym, zm = ellipsoid_plot_grid(center, radii, evecs, u, v)
xmm = patches_combined[:,0]; ymm = patches_combined[:,1]; zmm = patches_combined[:,2]

In [ ]:
# Run this if you want the patches and the fit to be shows as inside a single cube grid.
if False:
    xm  = [i%size for i in xm ]; ym  = [i%size for i in ym ]; zm  = [i%size for i in zm ]
    xmm = [i%size for i in xmm]; ymm = [i%size for i in ymm]; zmm = [i%size for i in zmm]

---

In [ ]:
u = np.linspace(0.0, 2.0 * np.pi, 400); v = np.linspace(0.0, np.pi, 400)   # We need to define them again since they were modified
plot_3d_interactive_ellipse2(xm, ym, zm, xmm, ymm, zmm)

In [ ]:
del xm, ym, zm; gc.collect()

---
---
---

## 11.4 Overlap of Ellipsoids Z=0.0

---

In [ ]:
if not os.path.exists(plots_path_0+"11.4___Ellipsoid_Overlap"): os.makedirs(plots_path_0+"11.4___Ellipsoid_Overlap")

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
uod_str = BASELINES[sz_indx]["uod_str"]
MK      = BASELINES[sz_indx]["MK"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

---

In [ ]:
file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
with open(file_path_ellips_Z+"Ellipsoid_values/l_center.pk", 'rb') as f: l_center = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_radii.pk",  'rb') as f: l_radii  = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_evecs.pk",  'rb') as f: l_evecs  = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_v.pk",      'rb') as f: l_v      = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_i1.pk",     'rb') as f: l_i1     = pkl.load(f)

In [ ]:
with open(file_path_MK+"fg___100.pk",  'rb') as f: fg_100 = pkl.load(f)
# Re-order the void indices so they start from 0 and no gaps.
fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())
no_voids = np.max(fg_100)+1   # +1 for the 0 index

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk",  'rb') as f: grid_fft = pkl.load(f)
vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)

---

In [ ]:
grid_elipses = np.zeros((size, size, size))

len_l_i1 = len(l_i1)
progress_bar(0, len_l_i1)
for i0, i1 in enumerate(l_i1):
    progress_bar(i0, len_l_i1)
    u = np.linspace(0.0, 2.0 * np.pi, 400); v = np.linspace(0.0, np.pi, 400)
    xm, ym, zm = ellipsoid_plot_grid(l_center[i0], l_radii[i0], l_evecs[i0], u, v)

    for i2 in range(len(xm)): grid_elipses[math.floor(xm[i2]%size)][math.floor(ym[i2]%size)][math.floor(zm[i2]%size)] = 1

with open(file_path_ellips_Z+"grid_elipses.pk", 'wb') as f: pkl.dump(grid_elipses, f)

---

In [ ]:
with open(file_path_ellips_Z+"grid_elipses.pk", 'rb') as f: grid_elipses = pkl.load(f)

---

In [ ]:
lll = 0
for ll in range(-20,20):
    lll += 1
    sc_i = sc + ll

    fig, axs = plt.subplots(figsize=(8,6), dpi=400)

    fig.suptitle(  r"Ellipsoid outline overlap" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")
    
    fg_100_sc = fg_100[sc_i].copy()
    walls_sc = np.ma.masked_where(fg_100_sc!=-2, fg_100_sc)
    walls_sc[walls_sc == -2] = 1
    
        
    imm = axs.imshow(fg_100_sc,                 vmin=0,   vmax=no_voids-1, cmap='binary',  extent=[0.0, 75, 0.0, 75])
    axs.imshow(walls_sc,                        vmin=0.1, vmax=2,          cmap='winter',  extent=[0.0, 75, 0.0, 75])
    axs.imshow(grid_elipses[sc_i], norm=LogNorm(vmin=0.1, vmax=1),         cmap='Purples', extent=[0.0, 75, 0.0, 75], alpha=0.6)


    divider = make_axes_locatable(axs)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
    cbar.set_label(r'Voids index (reordered)', rotation=270, labelpad=15)
    tks, tkss = set_ticks(0, no_voids-1, log_lin=False, int_if_possible=True)
    cbar.ax.set_yticks(tks, tkss)
    
    axs.set_xticks(range_75_5, range_75_5_tkss)
    axs.set_yticks(range_75_5, range_75_5_tkss)
    
    axs.set_xlabel('y (cMpc/h)')
    axs.set_ylabel('z (cMpc/h)')
        
    plt.tight_layout()
    plt.savefig(plots_path_0+"11.4___Ellipsoid_Overlap/"+str(lll)+".png")
    plt.close()

---
---
---

## 11.5 Density vs Radius

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"

---

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk",  'rb') as f: grid_fft = pkl.load(f)

In [ ]:
file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
with open(file_path_ellips_Z+"Ellipsoid_values/l_center.pk", 'rb') as f: l_center = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_radii.pk",  'rb') as f: l_radii  = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_evecs.pk",  'rb') as f: l_evecs  = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_v.pk",      'rb') as f: l_v      = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_i1.pk",     'rb') as f: l_i1     = pkl.load(f)
len_l_i1 = len(l_i1)

---
---
---

In [ ]:
l_densities = []
l_densities_min = []; l_densities_max = []; l_densities_mean = []
l_densities_all = [l_densities_min, l_densities_max, l_densities_mean]

l_distances_fft = []; l_distances_grid = []; l_distances_ellips = []
l_distances_all = [l_distances_fft, l_distances_grid, l_distances_ellips]

l_radii_max_fft = []; l_radii_max_grid = []; l_radii_max_ellips = []
l_radii_max_all = [l_radii_max_fft, l_radii_max_grid, l_radii_max_ellips]

centers_fft = []; centers_grid = []; centers_ellips = []
l_centers_all = [centers_fft, centers_grid, centers_ellips]

In [ ]:
for i0, i1 in enumerate(l_i1):
    
    with open(file_path_ellips_Z+"void_combined/"+str(i1)+".pk", 'rb') as f: void_combined = pkl.load(f)
    
    # Get the coordinates in the combined patch where grid_fft is at its minimum
    # The densities of these coordinates
    void_coords = void_combined%size
    densities = grid_fft[void_coords[:,0], void_coords[:,1], void_coords[:,2]]
    l_densities.append(densities)
    l_densities_min.append(np.min(densities)); l_densities_max.append(np.max(densities)); l_densities_mean.append(np.mean(densities))

    # The coordinates (in non-%size fashion) of the (first, but whatever, they are float) minimum in the void.
    # These are in the combined space, so can be <0 / >size-1
    center_fft    = void_combined[np.argmin(densities)]
    center_grid   = np.array([round(_,0) for _ in np.mean(void_combined, axis=0)])
    center_ellips = [int(round(_)) for _ in l_center[i0]]

    centers_fft.append(   center_fft)
    centers_grid.append(  center_grid)
    centers_ellips.append(center_ellips)

    # The distances from the minimum found above (from fft) and the ellipsoid fit center
    l_distances_fft.append(   np.sqrt(np.sum((void_combined - center_fft   )**2, axis=1)))
    l_distances_grid.append(  np.sqrt(np.sum((void_combined - center_grid  )**2, axis=1)))
    l_distances_ellips.append(np.sqrt(np.sum((void_combined - center_ellips)**2, axis=1)))

    l_radii_max_fft.append(   np.max(l_distances_fft[   -1]))
    l_radii_max_grid.append(  np.max(l_distances_grid[  -1]))
    l_radii_max_ellips.append(np.max(l_distances_ellips[-1]))

with open(file_path_ellips_Z+"l_densities.pk",     'wb') as f: pkl.dump(l_densities,     f)
with open(file_path_ellips_Z+"l_densities_all.pk", 'wb') as f: pkl.dump(l_densities_all, f)
with open(file_path_ellips_Z+"l_distances_all.pk", 'wb') as f: pkl.dump(l_distances_all, f)
with open(file_path_ellips_Z+"l_radii_max_all.pk", 'wb') as f: pkl.dump(l_radii_max_all, f)
with open(file_path_ellips_Z+"l_centers_all.pk",   'wb') as f: pkl.dump(l_centers_all,   f)#

---
---
---
---
---
---
---
---
---
---

# 12. Void Profiles

---
---
---

## 12.1 Compute

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"

---

Let $G$ be the set of grid cells belonging to the void, and let $E$ be the set of grid cells inside the fitted ellipsoid.
Their overlap is $G \cap E$, and their union is $G \cup E$.

In this comparison, $G$ is the voxelized void from the watershed grid, while $E$ is the voxelized version of the smooth fitted ellipsoid evaluated on the same grid.

---

The **coverage** measures what fraction of the original void grid is recovered by the ellipsoid.

A high coverage means that most of the void lies inside the ellipsoid. A low coverage means that the ellipsoid misses a large part of the void.

$$
\mathrm{coverage} = \frac{|G \cap E|}{|G|}
$$

---

The **purity** measures what fraction of the ellipsoid is actually occupied by the void grid:

A high purity means that most of the ellipsoid corresponds to real void cells. A low purity means that the ellipsoid includes a large amount of extra volume outside the void.

$$
\mathrm{purity} = \frac{|G \cap E|}{|E|}
$$

---

The **intersection-over-union** combines both effects into one overlap score:

$$
\mathrm{IoU} = \frac{|G \cap E|}{|G \cup E|} = \frac{|G \cap E|}{|G| + |E| - |G \cap E|}
$$

---

The **Dice coefficient** is another combined overlap score, giving twice the shared volume relative to the total size of both objects:

$$
\mathrm{Dice} = \frac{2|G \cap E|}{|G| + |E|}
$$

---
---
---

In [ ]:
def ellips_filling_fit(void_coords, center, radii, evecs, padding=2, evecs_are_columns=True):
    """
    Compare an unwrapped void grid with its fitted ellipsoid.

    INPUT
    void_coords        - (N, 3) integer coordinates of the void cells, preferably unwrapped
    center             - (3,) ellipsoid center
    radii              - (3,) ellipsoid semi-axis lengths
    evecs              - (3, 3) ellipsoid eigenvectors / principal axes
    padding            - extra cells around ellipsoid bounding box
    evecs_are_columns  - True if evecs[:,i] is axis i; False if evecs[i,:] is axis i

    OUTPUT
    coverage =  |G ∩ E| / |G|
    purity   =  |G ∩ E| / |E|
    IoU      =  |G ∩ E| / |G ∪ E|
    Dice     = 2|G ∩ E| / (|G| + |E|)
    """

    void_coords = np.asarray(void_coords, dtype=float)
    center      = np.asarray(center,      dtype=float)
    radii       = np.asarray(radii,       dtype=float)
    evecs       = np.asarray(evecs,       dtype=float)

    if len(void_coords) == 0: return np.nan, np.nan, np.nan, np.nan

    # Rotation:
        # - if evecs columns are principal axes: local coords = (x - center) @ evecs
        # - if evecs rows    are principal axes: local coords = (x - center) @ evecs.T
    R = evecs if evecs_are_columns else evecs.T

    def inside_ellipsoid(points):
        local = (points - center) @ R
        q = np.sum((local / radii)**2, axis=1)
        return q <= 1.0

    # G & E interseciton
    void_inside = inside_ellipsoid(void_coords)
    n_grid      = len(void_coords)
    n_overlap   = int(np.sum(void_inside))

    # voxelize the ellipsoid on the same grid
    half_widths = np.sum(np.abs(R) * radii[None, :], axis=1)

    mins = np.floor(center - half_widths - padding).astype(int)
    maxs = np.ceil( center + half_widths + padding).astype(int)
    
    
    # Chunk by x-slices
    YY, ZZ = np.meshgrid(np.arange(mins[1], maxs[1] + 1),
                         np.arange(mins[2], maxs[2] + 1), indexing="ij")
    yz = np.column_stack([YY.ravel(), ZZ.ravel()])

    n_ellips = 0
    for x in range(mins[0], maxs[0] + 1):
        xs = np.full((len(yz), 1), x)
        points = np.column_stack([xs, yz])
        n_ellips += int(np.sum(inside_ellipsoid(points)))

    
    # final ratios
    coverage = n_overlap / n_grid             if n_grid              > 0 else np.nan
    purity   = n_overlap / n_ellips           if n_ellips            > 0 else np.nan

    union = n_grid + n_ellips - n_overlap
    IoU   = n_overlap / union                 if union               > 0 else np.nan
    Dice  = 2*n_overlap / (n_grid + n_ellips) if (n_grid + n_ellips) > 0 else np.nan

    
    return coverage, purity, IoU, Dice

---

In [ ]:
file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
with open(file_path_ellips_Z+"Ellipsoid_values/l_center.pk", 'rb') as f: l_center = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_radii.pk",  'rb') as f: l_radii  = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_evecs.pk",  'rb') as f: l_evecs  = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_v.pk",      'rb') as f: l_v      = pkl.load(f)
with open(file_path_ellips_Z+"Ellipsoid_values/l_i1.pk",     'rb') as f: l_i1     = pkl.load(f)
len_l_i1 = len(l_i1)

In [ ]:
l_coverage = []; l_purity = []; l_IoU = []; l_Dice = []
l_volumes = []
progress_bar(0, len(l_i1)-1)
for i0, i1 in enumerate(l_i1):
    progress_bar(i0, len(l_i1)-1)
    
    with open(file_path_ellips_Z+"void_combined/"+str(i1)+".pk", 'rb') as f: void_combined = pkl.load(f)
    l_volumes.append(np.sum(void_combined))

    # Get the coordinates in the combined patch where grid_fft is at its minimum
    void_coords = void_combined%size

    #ratio_grid_overlap_ellips, ratio_ellips_overlap_grid = ellips_filling_fit(void_coords, ...)
    coverage, purity, IoU, Dice = ellips_filling_fit(void_combined, l_center[i0], l_radii[i0], l_evecs[i0])
    l_coverage.append(coverage); l_purity.append(purity); l_IoU.append(IoU); l_Dice.append(Dice)

In [ ]:
with open(file_path_ellips_Z+"l_coverage.pk", 'wb') as f: pkl.dump(l_coverage, f)
with open(file_path_ellips_Z+"l_purity.pk",   'wb') as f: pkl.dump(l_purity,   f)
with open(file_path_ellips_Z+"l_IoU.pk",      'wb') as f: pkl.dump(l_IoU,      f)
with open(file_path_ellips_Z+"l_Dice.pk",     'wb') as f: pkl.dump(l_Dice,     f)

---
---
---

In [ ]:
file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
with open(file_path_ellips_Z+"l_densities.pk",     'rb') as f: l_densities     = pkl.load(f)
with open(file_path_ellips_Z+"l_densities_all.pk", 'rb') as f: l_densities_all = pkl.load(f)
with open(file_path_ellips_Z+"l_distances_all.pk", 'rb') as f: l_distances_all = pkl.load(f)
with open(file_path_ellips_Z+"l_radii_max_all.pk", 'rb') as f: l_radii_max_all = pkl.load(f)
with open(file_path_ellips_Z+"l_centers_all.pk",   'rb') as f: l_centers_all   = pkl.load(f)

with open(file_path_ellips_Z+"l_coverage.pk",      'rb') as f: l_coverage      = pkl.load(f)
with open(file_path_ellips_Z+"l_purity.pk",        'rb') as f: l_purity        = pkl.load(f)
with open(file_path_ellips_Z+"l_IoU.pk",           'rb') as f: l_IoU           = pkl.load(f)
with open(file_path_ellips_Z+"l_Dice.pk",          'rb') as f: l_Dice          = pkl.load(f)

In [ ]:
len(l_densities), len(l_coverage), len(l_radii), len(l_centers_all[0])

---

In [ ]:
# keep only the profiles where the ellipsoids' center is inside the respective void
ellips_centers_inside = [ii[0] for ii in np.argwhere(np.array([np.min(_) for _ in l_distances_all[2]]) == 0.)]

l_densities     =  [l_densities[ _] for _ in ellips_centers_inside]
l_densities_all = [[l_densities1[_] for _ in ellips_centers_inside] for l_densities1 in l_densities_all]
l_distances_all = [[l_distances[ _] for _ in ellips_centers_inside] for l_distances  in l_distances_all]
l_radii_max_all = [[l_radii_max[ _] for _ in ellips_centers_inside] for l_radii_max  in l_radii_max_all]
l_centers_all   = [[l_centers[   _] for _ in ellips_centers_inside] for l_centers    in l_centers_all  ]

l_radii         =  [l_radii[     _] for _ in ellips_centers_inside]

l_coverage      =  [l_coverage[  _] for _ in ellips_centers_inside]
l_purity        =  [l_purity[    _] for _ in ellips_centers_inside]
l_IoU           =  [l_IoU[       _] for _ in ellips_centers_inside]
l_Dice          =  [l_Dice[      _] for _ in ellips_centers_inside]

In [ ]:
len(l_densities), len(l_coverage), len(l_radii), len(l_centers_all[0])

---

In [ ]:
l_radii_sss    = [np.sqrt(i**2+j**2+k**2) for [i,j,k] in l_radii]

In [ ]:
reasonable_radii = [int(_[0]) for _ in np.argwhere(np.array(l_radii_sss) <= 250)]   # 250 cells out of 512

l_densities     =  [l_densities[ _] for _ in reasonable_radii]
l_densities_all = [[l_densities1[_] for _ in reasonable_radii] for l_densities1 in l_densities_all]
l_distances_all = [[l_distances[ _] for _ in reasonable_radii] for l_distances  in l_distances_all]
l_radii_max_all = [[l_radii_max[ _] for _ in reasonable_radii] for l_radii_max  in l_radii_max_all]
l_centers_all   = [[l_centers[   _] for _ in reasonable_radii] for l_centers    in l_centers_all  ]

l_radii         =  [l_radii[     _] for _ in reasonable_radii]

l_coverage      =  [l_coverage[  _] for _ in reasonable_radii]
l_purity        =  [l_purity[    _] for _ in reasonable_radii]
l_IoU           =  [l_IoU[       _] for _ in reasonable_radii]
l_Dice          =  [l_Dice[      _] for _ in reasonable_radii]

In [ ]:
l_radii_sss    = [np.sqrt(i**2+j**2+k**2) for [i,j,k] in l_radii]

In [ ]:
len(l_densities), len(l_coverage), len(l_radii), len(l_centers_all[0])

---

In [ ]:
seelcted_ellips = [int(_[0]) for _ in np.argwhere(np.array(l_IoU) >= 0.6)]

lsel_densities     =  [l_densities[ _] for _ in seelcted_ellips]
lsel_densities_all = [[l_densities1[_] for _ in seelcted_ellips] for l_densities1 in l_densities_all]
lsel_distances_all = [[l_distances[ _] for _ in seelcted_ellips] for l_distances  in l_distances_all]
lsel_radii_max_all = [[l_radii_max[ _] for _ in seelcted_ellips] for l_radii_max  in l_radii_max_all]
lsel_centers_all   = [[l_centers[   _] for _ in seelcted_ellips] for l_centers    in l_centers_all  ]

lsel_radii         =  [l_radii[     _] for _ in seelcted_ellips]

lsel_coverage      =  [l_coverage[  _] for _ in seelcted_ellips]
lsel_purity        =  [l_purity[    _] for _ in seelcted_ellips]
lsel_IoU           =  [l_IoU[       _] for _ in seelcted_ellips]
lsel_Dice          =  [l_Dice[      _] for _ in seelcted_ellips]

In [ ]:
lsel_radii_sss = [np.sqrt(i**2+j**2+k**2) for [i,j,k] in lsel_radii]

In [ ]:
len(l_densities), len(l_coverage), len(l_radii), len(l_centers_all[0])

---

In [ ]:
if not os.path.exists(file_path_ellips_Z+"filtered_vals/"): os.makedirs(file_path_ellips_Z+"filtered_vals/")

In [ ]:
with open(file_path_ellips_Z+"filtered_vals/lsel_densities.pk"    , "wb") as f: pkl.dump(lsel_densities    , f)
with open(file_path_ellips_Z+"filtered_vals/lsel_densities_all.pk", "wb") as f: pkl.dump(lsel_densities_all, f)
with open(file_path_ellips_Z+"filtered_vals/lsel_distances_all.pk", "wb") as f: pkl.dump(lsel_distances_all, f)
with open(file_path_ellips_Z+"filtered_vals/lsel_radii_max_all.pk", "wb") as f: pkl.dump(lsel_radii_max_all, f)
with open(file_path_ellips_Z+"filtered_vals/lsel_centers_all.pk",   "wb") as f: pkl.dump(lsel_centers_all,   f)
with open(file_path_ellips_Z+"filtered_vals/lsel_radii.pk"        , "wb") as f: pkl.dump(lsel_radii        , f)
with open(file_path_ellips_Z+"filtered_vals/lsel_coverage.pk"     , "wb") as f: pkl.dump(lsel_coverage     , f)
with open(file_path_ellips_Z+"filtered_vals/lsel_purity.pk"       , "wb") as f: pkl.dump(lsel_purity       , f)
with open(file_path_ellips_Z+"filtered_vals/lsel_IoU.pk"          , "wb") as f: pkl.dump(lsel_IoU          , f)
with open(file_path_ellips_Z+"filtered_vals/lsel_Dice.pk"         , "wb") as f: pkl.dump(lsel_Dice         , f)
with open(file_path_ellips_Z+"filtered_vals/lsel_radii_sss.pk"    , "wb") as f: pkl.dump(lsel_radii_sss    , f)

with open(file_path_ellips_Z+"filtered_vals/l_densities.pk"    ,    "wb") as f: pkl.dump(l_densities    ,    f)
with open(file_path_ellips_Z+"filtered_vals/l_densities_all.pk",    "wb") as f: pkl.dump(l_densities_all,    f)
with open(file_path_ellips_Z+"filtered_vals/l_distances_all.pk",    "wb") as f: pkl.dump(l_distances_all,    f)
with open(file_path_ellips_Z+"filtered_vals/l_radii_max_all.pk",    "wb") as f: pkl.dump(l_radii_max_all,    f)
with open(file_path_ellips_Z+"filtered_vals/l_centers_all.pk",      "wb") as f: pkl.dump(l_centers_all,      f)
with open(file_path_ellips_Z+"filtered_vals/l_radii.pk"        ,    "wb") as f: pkl.dump(l_radii        ,    f)
with open(file_path_ellips_Z+"filtered_vals/l_coverage.pk"     ,    "wb") as f: pkl.dump(l_coverage     ,    f)
with open(file_path_ellips_Z+"filtered_vals/l_purity.pk"       ,    "wb") as f: pkl.dump(l_purity       ,    f)
with open(file_path_ellips_Z+"filtered_vals/l_IoU.pk"          ,    "wb") as f: pkl.dump(l_IoU          ,    f)
with open(file_path_ellips_Z+"filtered_vals/l_Dice.pk"         ,    "wb") as f: pkl.dump(l_Dice         ,    f)
with open(file_path_ellips_Z+"filtered_vals/l_radii_sss.pk"    ,    "wb") as f: pkl.dump(l_radii_sss    ,    f)

---
---
---

In [ ]:
with open(file_path_ellips_Z+"filtered_vals/lsel_centers_all.pk",   "rb") as f: lsel_centers_all   = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_centers_all.pk",      "rb") as f: l_centers_all      = pkl.load(f)

In [ ]:
centers_diff_sqrt_fg = []; centers_diff_sqrt_eg = []
for i0 in range(len(l_centers_all[0])):
    x0,y0,z0 = l_centers_all[0][   i0]
    x1,y1,z1 = l_centers_all[1][  i0]
    x2,y2,z2 = l_centers_all[2][i0]

    centers_diff_sqrt_fg.append(np.sqrt(np.min([np.abs((x0-x1)%size), np.abs((x1-x0)%size)])**2+
                                        np.min([np.abs((y0-y1)%size), np.abs((y1-y0)%size)])**2+
                                        np.min([np.abs((z0-z1)%size), np.abs((z1-z0)%size)])**2))
    centers_diff_sqrt_eg.append(np.sqrt(np.min([np.abs((x2-x1)%size), np.abs((x1-x2)%size)])**2+
                                        np.min([np.abs((y2-y1)%size), np.abs((y1-y2)%size)])**2+
                                        np.min([np.abs((z2-z1)%size), np.abs((z1-z2)%size)])**2))


centers_diff_sqrt_fg_sel = []; centers_diff_sqrt_eg_sel = []
for i0 in range(len(lsel_centers_all[0])):
    x0,y0,z0 = lsel_centers_all[0][   i0]
    x1,y1,z1 = lsel_centers_all[1][  i0]
    x2,y2,z2 = lsel_centers_all[2][i0]

    centers_diff_sqrt_fg_sel.append(np.sqrt(np.min([np.abs((x0-x1)%size), np.abs((x1-x0)%size)])**2+
                                            np.min([np.abs((y0-y1)%size), np.abs((y1-y0)%size)])**2+
                                            np.min([np.abs((z0-z1)%size), np.abs((z1-z0)%size)])**2))
    centers_diff_sqrt_eg_sel.append(np.sqrt(np.min([np.abs((x2-x1)%size), np.abs((x1-x2)%size)])**2+
                                            np.min([np.abs((y2-y1)%size), np.abs((y1-y2)%size)])**2+
                                            np.min([np.abs((z2-z1)%size), np.abs((z1-z2)%size)])**2))

In [ ]:
both_centers_diff_sqrt_fg = [centers_diff_sqrt_fg, centers_diff_sqrt_fg_sel]
both_centers_diff_sqrt_eg = [centers_diff_sqrt_eg, centers_diff_sqrt_eg_sel]

In [ ]:
max_cells = 100

fig, ax = plt.subplots(1, 2, figsize=(12, 3), dpi=200)

fig.suptitle(  r"Comparison of the spatial difference between the different types of void origins " + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]" + "\n", y=1.275)

ax[0].set_title("All ellipsoid-fitted voids")
ax[1].set_title("Selected ellipsoid-fitted voids")

for ii0 in range(2):
    
    u0 = ax[ii0].hist(both_centers_diff_sqrt_fg[ii0], bins=np.linspace(0, max_cells, max_cells+1), density=True, color=(1, 0, 0, 0.4), edgecolor="red", linewidth=1.2, label=r"minimum density - void average")
    u1 = ax[ii0].hist(both_centers_diff_sqrt_eg[ii0], bins=np.linspace(0, max_cells, max_cells+1), density=True, color=(0, 0, 1, 0.4), edgecolor="blue", linewidth=1.2, label=r"ellipsoid center - void average")
    
    max_hist = np.max([np.max(u0[0]), np.max(u1[0])])
    ytks, ytkss = set_ticks(0, max_hist, log_lin=False, int_if_possible=False, float_g=True, g=3)
    ax[ii0].set_yticks(ytks, ytkss)
    
    xtks, xtkss = set_ticks(0, max_cells, log_lin=False, int_if_possible=True)
    ax[ii0].set_xticks(xtks, xtkss)
    ax[ii0].set_xlabel("distance [cells]")
    
    ax_top = ax[ii0].twiny()
    ax_top.set_xlim(ax[ii0].get_xlim())
    #xtkss_top = [latex_float(x, int_if_possible=True, float_g=1) for x in np.array(xtks)*d_xyz]
    ax_top.set_xticks(xtks, [round(_,1) for _ in np.array(xtks)*d_xyz])
    ax_top.set_xlabel(r"distance [cMpc/$h$]")
    
    ax[ii0].grid()

ax[0].set_ylabel("(Normalized) count density of centers")
ax[1].legend()

fig.savefig(plots_path_0+"12.1___difference_in_centers.png", bbox_inches="tight", pad_inches=0.25)
plt.close()

---

In [ ]:
with open(file_path_ellips_Z+"filtered_vals/lsel_coverage.pk"     , "rb") as f: lsel_coverage      = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_purity.pk"       , "rb") as f: lsel_purity        = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_IoU.pk"          , "rb") as f: lsel_IoU           = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_radii_sss.pk"    , "rb") as f: lsel_radii_sss     = pkl.load(f)

with open(file_path_ellips_Z+"filtered_vals/l_coverage.pk"     ,    "rb") as f: l_coverage         = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_purity.pk"       ,    "rb") as f: l_purity           = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_IoU.pk"          ,    "rb") as f: l_IoU              = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_radii_sss.pk"    ,    "rb") as f: l_radii_sss        = pkl.load(f)

In [ ]:
from matplotlib.patches import Rectangle

fig, axs = plt.subplots(3,2, figsize=(12,12), dpi=200)

fig.suptitle(  r"Coverage, purity and IoU vs. max radii: testing the ellipsoid fits" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]" + "\n")

axs[0,0].set_title(r"All ellipsoid fits",                                  fontsize=10)
axs[0,1].set_title(r"Ellipsoid fits that pass threshold:  IoU$\geq 0.6$",  fontsize=10)

plot_data = [[l_radii_sss, l_coverage, (30,30), lsel_radii_sss, lsel_coverage, (10,10), r"Coverage = $\frac{|G \cap E|}{|G|}$"],
             [l_radii_sss, l_purity,   (30,30), lsel_radii_sss, lsel_purity,   (10,10), r"Purity = $\frac{|G \cap E|}{|E|}$"],
             [l_radii_sss, l_IoU,      (30,30), lsel_radii_sss, lsel_IoU,      (10,10), r"IoU = $\frac{|G \cap E|}{|G \cup E|}$"],]

for i3 in range(3):

    full_radii = np.array(plot_data[i3][0].copy())
    sel_radii  = np.array(plot_data[i3][3].copy())
    sel_vals   = np.array(plot_data[i3][4].copy())

    full_no_bins_x, full_no_bins_y = plot_data[i3][2]

    full_xmin, full_xmax = 0, np.max(full_radii)
    full_ymin, full_ymax = 0, 1

    zoom_xmin, zoom_xmax = 0, np.max(sel_radii)
    zoom_ymin, zoom_ymax = np.min(sel_vals), np.max(sel_vals)

    if zoom_ymin == zoom_ymax:
        zoom_ymin -= 0.5
        zoom_ymax += 0.5

    for i0 in range(2):

        if i0 == 0:
            l_radii = np.array(plot_data[i3][0].copy())
            l_vals  = np.array(plot_data[i3][1].copy())
            bins    = plot_data[i3][2]
        else:
            l_radii = np.array(plot_data[i3][3].copy())
            l_vals  = np.array(plot_data[i3][4].copy())
            bins    = plot_data[i3][5]

        no_bins_x, no_bins_y = bins

        max_radii_cells = np.max(l_radii)
        max_radii_phys  = max_radii_cells / size * 75

        if i0 == 0:               ymin_val  = 0;              ymax_val  = 1
        else:                     ymin_val  = np.min(l_vals); ymax_val  = np.max(l_vals)

        if ymin_val == ymax_val:  ymin_val -= 0.5;            ymax_val += 0.5

        density_2d, xedges, yedges = np.histogram2d(l_radii, l_vals, bins=bins, range=[[0, max_radii_cells], [ymin_val, ymax_val]])
        density_2d = density_2d.T

        vmin0 = 1
        vmax0 = np.max(density_2d)

        imm = axs[i3,i0].imshow(np.ma.masked_less_equal(density_2d, 0.1),
                                norm=LogNorm(vmin=vmin0, vmax=vmax0),
                                cmap='winter',
                                aspect='auto',
                                origin='lower',
                                extent=[0, no_bins_x, 0, no_bins_y])

        axs[i3,i0].set_box_aspect(no_bins_y/no_bins_x)
        axs[i3,i0].set_xlim(0, no_bins_x)
        axs[i3,i0].set_ylim(0, no_bins_y)

        divider = make_axes_locatable(axs[i3,i0])
        cax     = divider.append_axes('right', size='3%', pad=-0.3)
        cbar    = plt.colorbar(imm, cax=cax, orientation='vertical')

        tks = np.arange(int(vmin0), int(vmax0) + 1)

        cbar.locator = FixedLocator(tks)
        cbar.formatter = FixedFormatter([str(_) for _ in tks])
        cbar.update_ticks()

        cbar.ax.yaxis.set_minor_locator(NullLocator())
        cbar.ax.yaxis.set_minor_formatter(NullFormatter())

        ccc = divider.append_axes('right', size='3%', pad=1.)
        ccc.set_xticks([], [])
        ccc.set_yticks([], [])
        ccc.axis('off')

        cbar.set_label('     Void counts', rotation=270, labelpad=15)

        if i0 == 0:
            tks, tkss = set_ticks(ymin_val, ymax_val, log_lin=False, int_if_possible=True, float_g=True, g=3, number_of_tks_clean_min=3)
            axs[i3,i0].set_yticks([(_ - ymin_val)/(ymax_val - ymin_val)*no_bins_y for _ in tks], tkss)

            tks, tkss = set_ticks(0, max_radii_phys, log_lin=False, int_if_possible=True, float_asitis_ends=True)
            axs[i3,i0].set_xticks([_/max_radii_phys*no_bins_x for _ in tks], tkss)

        else:
            x_tick_pos = np.arange(no_bins_x) + 0.5
            y_tick_pos = np.arange(no_bins_y) + 0.5

            x_tick_vals = ((xedges[:-1] + xedges[1:]) / 2) / size * 75
            y_tick_vals =  (yedges[:-1] + yedges[1:]) / 2

            axs[i3,i0].set_xticks(x_tick_pos, [f"{_: .1f}" for _ in x_tick_vals], rotation=45, ha='right')
            axs[i3,i0].set_yticks(y_tick_pos, [f"{_:.3g}" for _ in y_tick_vals])

        axs[i3,i0].grid(alpha=0.3)

    x0 = (zoom_xmin-full_xmin) / (full_xmax-full_xmin) * full_no_bins_x
    x1 = (zoom_xmax-full_xmin) / (full_xmax-full_xmin) * full_no_bins_x
    y0 = (zoom_ymin-full_ymin) / (full_ymax-full_ymin) * full_no_bins_y
    y1 = (zoom_ymax-full_ymin) / (full_ymax-full_ymin) * full_no_bins_y
    axs[i3,0].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor=(0.5, 0.5, 0.5, 0.18), edgecolor=(0.2, 0.2, 0.2, 0.9), linewidth=0.5, zorder=10))

for i0 in range(3):
    axs[i0,0].set_ylabel(plot_data[i0][6], labelpad=15)
    axs[i0,1].add_patch(Rectangle((0, 0), 1, 1, transform=axs[i0,1].transAxes, facecolor=(0.5, 0.5, 0.5, 0.12), edgecolor=(0.2, 0.2, 0.2, 0.9), linewidth=1.2, zorder=10, clip_on=False))

axs[2,0].set_xlabel(r"$R_i$ [cMpc/h]")
axs[2,1].set_xlabel(r"$R_i$ [cMpc/h]")

plt.tight_layout()
plt.savefig(plots_path_0+"12.1___coverage_purity_IoU__vs_maxradii.png")
plt.close()

---
---
---

## 12.2 Testing the ellipsoid fit and density analysis

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"

---

In [ ]:
file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
with open(file_path_ellips_Z+"filtered_vals/l_densities.pk"    ,    "rb") as f: l_densities     = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_densities_all.pk",    "rb") as f: l_densities_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_distances_all.pk",    "rb") as f: l_distances_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_radii_max_all.pk",    "rb") as f: l_radii_max_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_radii.pk"        ,    "rb") as f: l_radii         = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_coverage.pk"     ,    "rb") as f: l_coverage      = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_purity.pk"       ,    "rb") as f: l_purity        = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_IoU.pk"          ,    "rb") as f: l_IoU           = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_Dice.pk"         ,    "rb") as f: l_Dice          = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l_radii_sss.pk"    ,    "rb") as f: l_radii_sss     = pkl.load(f)

---

In [ ]:
# The code below uses bins for density values >= 0... which in the delta+1 normalization, they must be.
density_2d_min = []; density_2d_max = []; density_2d_mean = []
density_2d_all = [density_2d_min, density_2d_max, density_2d_mean]

d_yx_all = [[], [], []]

no_bins_y = 29; no_bins_x = 43

In [ ]:
# Min Max Mean
l_densities_mean = l_densities_all[2]
for i3 in range(3):
    l_densities = np.array(l_densities_all[i3].copy())
    if i3 < 2: l_densities /= np.array(l_densities_mean)

    # fft grid and ellips
    for i0 in range(3):
        l_radii_max = l_radii_max_all[i0]

        # indx 0  for origin
        # indx -1 for the max
        # anything in between for values that fall between 0-1%, 1-2%... 99-100% (as an example in the case no_bins is 100 for x or y)
        density_2d_all[i3].append(np.zeros((no_bins_y+2, no_bins_x+2)))

        max_l_densities = np.max(l_densities)
        #max_l_radii_max = np.max(l_radii_max)
        max_l_radii_max = 307
        
        d_y = max_l_densities/no_bins_y
        d_x = max_l_radii_max/no_bins_x
        d_yx_all[i3].append([d_y, d_x])
        
        for i1 in range(len(l_densities)):
            
            if l_radii_max[i1] < max_l_radii_max:
    
                if   l_radii_max[i1] == 0:               indx_x =  0
                elif l_radii_max[i1] == max_l_radii_max: indx_x = -1
                else:
                    indx_x = int(l_radii_max[i1]//d_x)+1
                    if indx_x == no_bins_x+1: indx_x = no_bins_x
    
                if   l_densities[i1] == 0:               indx_y =  0
                elif l_densities[i1] == max_l_densities: indx_y = -1
                else:
                    indx_y = int(l_densities[i1]//d_y)+1
                    if indx_y == no_bins_y+1: indx_y = no_bins_y
            
            density_2d_all[i3][-1][indx_y][indx_x] += 1
        density_2d_all[    i3][-1][density_2d_all[i3][-1] == 0] = 0.1

---

In [ ]:
len(l_densities_all[1])

In [ ]:
no_bins_y

In [ ]:
fig, axs = plt.subplots(3,3, figsize=(12,8), dpi=200)

fig.suptitle(  r"(Normed) minimum, maximum and mean densities inside a individual voids vs. their maximum radius" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")

axs[0,0].set_title("Minimum void density center\n",  fontsize=10)
axs[0,1].set_title("Void coordinates mean center\n", fontsize=10)
axs[0,2].set_title("Fitted ellipsoid center\n",      fontsize=10)

# Min Max Mean
for i3 in range(3):
    
    l_densities = np.array(l_densities_all[i3].copy())
    if i3 < 2: l_densities /= np.array(l_densities_mean)

    # fft grid and ellips
    for i0 in range(3):

        density_2d  = density_2d_all[i3][i0].copy()
        d_yx        = d_yx_all[      i3][i0]
        l_radii_max = l_radii_max_all[   i0]

        #density_2d[density_2d <= 0] = 0.1
        #density_2d = smooth_non_cyclic(density_2d, sigma=1)#.1*no_bins_scaledx/100)
        #density_2d[density_2d <= 0] = 0.1
        
        vmin0 = 1; vmax0 = np.max(density_2d)
        imm = axs[i3,i0].imshow(np.ma.masked_equal(density_2d, 0.1), norm=LogNorm(vmin=vmin0, vmax=vmax0), cmap='winter', aspect='auto')
        axs[i3,i0].set_box_aspect(len(density_2d)/len(density_2d[0]))
    
        divider   = make_axes_locatable(axs[i3,i0])
        cax       = divider.append_axes('right', size='3%', pad=0.1)
        cbar      = plt.colorbar(imm, cax=cax, orientation='vertical')
        cbar.ax.set_yticklabels([])
        if vmin0 != vmax0: tks, tkss = set_ticks(vmin0, vmax0, int_if_possible=True, float_g=True, number_of_tks_clean_max=6)
        else:              tks, tkss = [[vmin0], [str(vmin0)]]
        cbar.ax.set_yticks(tks, tkss)
        cbar.locator = FixedLocator(tks)
        cbar.formatter = FixedFormatter(tkss)
        cbar.update_ticks()
        
        cbar.ax.yaxis.set_minor_locator(NullLocator())
        cbar.ax.yaxis.set_minor_formatter(NullFormatter())
        ccc = divider.append_axes('right', size='3%', pad=0.3)
        ccc.set_xticks([], []); ccc.set_yticks([], []); ccc.axis('off')
        if i0 == 1: cbar.set_label('     Void counts', rotation=270, labelpad=8)
    
        

        tks, tkss = set_ticks(0, np.max(l_densities), log_lin=False, int_if_possible=True, float_g=True, g=3)
        axs[i3,i0].set_yticks([_/np.max(l_densities)*len(density_2d) for _ in tks], tkss)
        axs[i3,i0].set_ylim(0, no_bins_y)

        tks, tkss = set_ticks(0, np.max(max_l_radii_max/size*75), log_lin=False, int_if_possible=True, float_asitis_ends=True)
        axs[i3,i0].set_xticks([_/np.max(max_l_radii_max/size*75)*len(density_2d[0]) for _ in tks], tkss)
        axs[i3,i0].set_xlim(0, no_bins_x-5)
        
        
        axs[i3,i0].grid(alpha=0.3)
        

axs[0,0].set_ylabel(r"$\frac{\delta_{min,i} + 1}{\overline{\delta}_i + 1}$", labelpad=3, fontsize=14)
axs[1,0].set_ylabel(r"$\frac{\delta_{max,i} + 1}{\overline{\delta}_i + 1}$", labelpad=3, fontsize=14)
axs[2,0].set_ylabel(r"$\overline{\delta_i} + 1$",                            labelpad=3)
axs[2,0].set_xlabel(r"$R_i$ [cMpc/h]")
axs[2,1].set_xlabel(r"$R_i$ [cMpc/h]")
axs[2,2].set_xlabel(r"$R_i$ [cMpc/h]")


plt.tight_layout()
plt.savefig(plots_path_0+"12.2___radius_max___minmaxmean___fft_ellips.png")
plt.close()

---

In [ ]:
fig, axs = plt.subplots(dpi=400, figsize=(12,5))

fig.suptitle(r"Coverage, purity and IoU: distribution of scores" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")

for vals, color, label in [(l_coverage, "C0", r"Coverage = $\frac{|G \cap E|}{|G|}$"),
                           (l_purity,   "C1", r"Purity = $\frac{|G \cap E|}{|E|}$"),
                           (l_IoU,      "C2", r"IoU = $\frac{|G \cap E|}{|G \cup E|}$")]:
    
    plt.hist(vals, bins=np.linspace(0, 1, 51), alpha=0.25, color=color, label=label)
    plt.hist(vals, bins=np.linspace(0, 1, 51), histtype="step", linewidth=1, color=color)

plt.axvline(0.6, lw=2, c="r", label=r"IoU$\geq 0.6$ threshold")

plt.xticks([0.1*_ for _ in range(11)])
plt.xlabel("score")
plt.ylabel("count")
plt.legend()

plt.grid(lw=1.3, c="grey", alpha=0.7, zorder=5)

plt.tight_layout()
plt.savefig(plots_path_0+"12.2___coverage_purity_IoU___scores.png")
plt.close()

---
---
---

In [ ]:
with open(file_path_ellips_Z+"filtered_vals/lsel_densities.pk"    , "rb") as f: l_densities     = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_densities_all.pk", "rb") as f: l_densities_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_distances_all.pk", "rb") as f: l_distances_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_radii_max_all.pk", "rb") as f: l_radii_max_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_radii.pk"        , "rb") as f: l_radii         = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_coverage.pk"     , "rb") as f: l_coverage      = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_purity.pk"       , "rb") as f: l_purity        = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_IoU.pk"          , "rb") as f: l_IoU           = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_Dice.pk"         , "rb") as f: l_Dice          = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/lsel_radii_sss.pk"    , "rb") as f: l_radii_sss     = pkl.load(f)

---

In [ ]:
# The code below uses bins for density values >= 0... which in the delta+1 normalization, they must be.
density_2d_min = []; density_2d_max = []; density_2d_mean = []
density_2d_all = [density_2d_min, density_2d_max, density_2d_mean]

d_yx_all = [[], [], []]

no_bins_y = 29; no_bins_x = 43

In [ ]:
# Min Max Mean
l_densities_mean = l_densities_all[2]
for i3 in range(3):
    l_densities = np.array(l_densities_all[i3].copy())
    if i3 < 2: l_densities /= np.array(l_densities_mean)

    # fft grid and ellips
    for i0 in range(3):
        l_radii_max = l_radii_max_all[i0]

        # indx 0  for origin
        # indx -1 for the max
        # anything in between for values that fall between 0-1%, 1-2%... 99-100% (as an example in the case no_bins is 100 for x or y)
        density_2d_all[i3].append(np.zeros((no_bins_y+2, no_bins_x+2)))

        max_l_densities = np.max(l_densities)
        #max_l_radii_max = np.max(l_radii_max)
        max_l_radii_max = 307
        
        d_y = max_l_densities/no_bins_y
        d_x = max_l_radii_max/no_bins_x
        d_yx_all[i3].append([d_y, d_x])
        
        for i1 in range(len(l_densities)):
            
            if l_radii_max[i1] < max_l_radii_max:
    
                if   l_radii_max[i1] == 0:               indx_x =  0
                elif l_radii_max[i1] == max_l_radii_max: indx_x = -1
                else:
                    indx_x = int(l_radii_max[i1]//d_x)+1
                    if indx_x == no_bins_x+1: indx_x = no_bins_x
    
                if   l_densities[i1] == 0:               indx_y =  0
                elif l_densities[i1] == max_l_densities: indx_y = -1
                else:
                    indx_y = int(l_densities[i1]//d_y)+1
                    if indx_y == no_bins_y+1: indx_y = no_bins_y
            
            density_2d_all[i3][-1][indx_y][indx_x] += 1
        density_2d_all[    i3][-1][density_2d_all[i3][-1] == 0] = 0.1

In [ ]:
len(l_densities_all[1])

---

In [ ]:
fig, axs = plt.subplots(3,3, figsize=(12,8), dpi=200)

fig.suptitle(  r"(Normed) minimum, maximum and mean densities inside a individual voids vs. their maximum radius" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")

axs[0,0].set_title("Minimum void density center\n",  fontsize=10)
axs[0,1].set_title("Void coordinates mean center\n", fontsize=10)
axs[0,2].set_title("Fitted ellipsoid center\n",      fontsize=10)

# Min Max Mean
for i3 in range(3):
    
    l_densities = np.array(l_densities_all[i3].copy())
    if i3 < 2: l_densities /= np.array(l_densities_mean)

    # fft grid and ellips
    for i0 in range(3):

        density_2d  = density_2d_all[i3][i0].copy()
        d_yx        = d_yx_all[      i3][i0]
        l_radii_max = l_radii_max_all[   i0]

        #density_2d[density_2d <= 0] = 0.1
        #density_2d = smooth_non_cyclic(density_2d, sigma=1)#.1*no_bins_scaledx/100)
        #density_2d[density_2d <= 0] = 0.1
        
        vmin0 = 1; vmax0 = np.max(density_2d)
        imm = axs[i3,i0].imshow(np.ma.masked_equal(density_2d, 0.1), norm=LogNorm(vmin=vmin0, vmax=vmax0), cmap='winter', aspect='auto')
        axs[i3,i0].set_box_aspect(len(density_2d)/len(density_2d[0]))
    
        divider   = make_axes_locatable(axs[i3,i0])
        cax       = divider.append_axes('right', size='3%', pad=0.1)
        cbar      = plt.colorbar(imm, cax=cax, orientation='vertical')
        cbar.ax.set_yticklabels([])
        if vmin0 != vmax0: tks, tkss = set_ticks(vmin0, vmax0, int_if_possible=True, float_g=True, number_of_tks_clean_max=6)
        else:              tks, tkss = [[vmin0], [str(vmin0)]]
        cbar.ax.set_yticks(tks, tkss)
        cbar.locator = FixedLocator(tks)
        cbar.formatter = FixedFormatter(tkss)
        cbar.update_ticks()
        
        cbar.ax.yaxis.set_minor_locator(NullLocator())
        cbar.ax.yaxis.set_minor_formatter(NullFormatter())
        ccc = divider.append_axes('right', size='3%', pad=0.3)
        ccc.set_xticks([], []); ccc.set_yticks([], []); ccc.axis('off')
        if i0 == 1: cbar.set_label('     Void counts', rotation=270, labelpad=8)
    
        

        tks, tkss = set_ticks(0, np.max(l_densities), log_lin=False, int_if_possible=True, float_g=True, g=3)
        axs[i3,i0].set_yticks([_/np.max(l_densities)*len(density_2d) for _ in tks], tkss)
        axs[i3,i0].set_ylim(0, no_bins_y)

        tks, tkss = set_ticks(0, np.max(max_l_radii_max/size*75), log_lin=False, int_if_possible=True, float_asitis_ends=True)
        axs[i3,i0].set_xticks([_/np.max(max_l_radii_max/size*75)*len(density_2d[0]) for _ in tks], tkss)
        axs[i3,i0].set_xlim(0, no_bins_x-5)
        
        
        axs[i3,i0].grid(alpha=0.3)
        

axs[0,0].set_ylabel(r"$\frac{\delta_{min,i} + 1}{\overline{\delta}_i + 1}$", labelpad=3, fontsize=14)
axs[1,0].set_ylabel(r"$\frac{\delta_{max,i} + 1}{\overline{\delta}_i + 1}$", labelpad=3, fontsize=14)
axs[2,0].set_ylabel(r"$\overline{\delta_i} + 1$",                            labelpad=3)
axs[2,0].set_xlabel(r"$R_i$ [cMpc/h]")
axs[2,1].set_xlabel(r"$R_i$ [cMpc/h]")
axs[2,2].set_xlabel(r"$R_i$ [cMpc/h]")


plt.tight_layout()
plt.savefig(plots_path_0+"12.2___radius_max___minmaxmean___fft_ellips_select.png")
plt.close()

---
---
---
---
---
---
---
---
---
---

# 13. Density profiles

---
---
---

## 13.1 Compute

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod

---

In [ ]:
sel = "sel"
#sel = ""

In [ ]:
file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_densities.pk"    ,    "rb") as f: l_densities     = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_densities_all.pk",    "rb") as f: l_densities_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_distances_all.pk",    "rb") as f: l_distances_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_radii_max_all.pk",    "rb") as f: l_radii_max_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_radii.pk"        ,    "rb") as f: l_radii         = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_coverage.pk"     ,    "rb") as f: l_coverage      = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_purity.pk"       ,    "rb") as f: l_purity        = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_IoU.pk"          ,    "rb") as f: l_IoU           = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_Dice.pk"         ,    "rb") as f: l_Dice          = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_radii_sss.pk"    ,    "rb") as f: l_radii_sss     = pkl.load(f)

---
---
---

In [ ]:
min_density = -1   # the minimum on the log scale we consider meaningful

---

### Dealing with the x-axis bins

In [ ]:
@njit
def all_distances_fct(x,g):

    # we found that, for our max distances, 4 decimals suffice to produce all values

    all_distances = set()
    for i in range(x):
        for j in range(x):
            for k in range(x):
                all_distances.add(round(np.sqrt(i**2 + j**2 + k**2),g))
    all_distances = list(all_distances)
    all_distances.sort()
    return all_distances

In [ ]:
def create_new_lists0(x, y, dx):

    '''
    This function takes the list of densities at the exact distances in the grid and the respective list of distances and outputs:
    - a list of densities at the distances interpolated over intervals dx (using their mean), where if inside such an interval there 
        is a single density value, it returns that exact value (ofc), but also it sets it at its precise location
    - the corresponding distance list
    '''

    len_x = len(x)
    
    x_array = np.array(x)
    d_array_dxs = x_array[1:]-x_array[:-1]
        
    x_new = []; y_new = []
    
    x_start = 0; i = 0
    while i < len_x:
        k = math.floor((x[i] - x_start)/dx)
        
        j = i
        while j < len_x and math.floor((x[j]-x_start)/dx) == k: j += 1
        
        if j > i:
            mean_y = np.mean(y[i:j])
            if j-i == 1: bin_start = x[i]
            else:        bin_start = x_start + (k+0.5) * dx
            x_new.append(round(bin_start,4)); y_new.append(mean_y)
        i = j
    
    return x_new, y_new

---

In [ ]:
roundingterm = 4
max_radii_all     = [np.max([np.max(_) for _ in ld]) for ld in l_distances_all]   # max radius among all 3 cases
all_distances_all = [all_distances_fct(_,roundingterm) for _ in max_radii_all]    # creating lists of all cells distances for each max radius

print(max_radii_all)

In [ ]:
factor_x = 2   # from radius 0 to the max radius, we introduce more bins... so for max radius e.g. 63.2 with
               #    a scale factor 2, we introduce 127 bins (thus the scaling nomenclature)

no_bins_nonscaledx_all = [math.ceil(_*factor_x) for _ in max_radii_all]
no_bins_scaledx_all    = no_bins_nonscaledx_all   # we chose to keep the same number of bins in the scaled case too

bins_nonscaledx_all    = [np.linspace(0, _+1, (_+1)*10+1) for _ in no_bins_nonscaledx_all]
bins_scaledx_all       = [np.linspace(0, _,    _   *10+1) for _ in no_bins_scaledx_all]

print(no_bins_nonscaledx_all)

In [ ]:
with open(file_path_ellips_Z+"bins_nonscaledx_all.pk",    'wb') as f: pkl.dump(bins_nonscaledx_all,    f)
with open(file_path_ellips_Z+"bins_scaledx_all.pk",       'wb') as f: pkl.dump(bins_scaledx_all,       f)
with open(file_path_ellips_Z+"no_bins_nonscaledx_all.pk", 'wb') as f: pkl.dump(no_bins_nonscaledx_all, f)
with open(file_path_ellips_Z+"no_bins_scaledx_all.pk",    'wb') as f: pkl.dump(no_bins_scaledx_all,    f)

---

### Dealing with the y-axis 

In [ ]:
no_bins_scaledx_all[1]*10+1

In [ ]:
no_pixels_y = 4000

densities_nonscaledx_nonscaledy = []
densities_scaledx_scaledy       = []

# we only produce this for the grid avg and ellipsoid center definition.... spoiler altert: otherwise, your profiles are obviously not centered
pixels_scaledx_scaledy          = [np.zeros((no_pixels_y+1, no_bins_scaledx_all[_+1]*10+1)) for _ in range(2)]
pixels_nonscaledx_nonscaledy    = [np.zeros((no_pixels_y+1, len(bins_nonscaledx_all[_+1]))) for _ in range(2)]

---

### Running the loop

In [ ]:
dx = 1.0

pixel_scalefactor = no_pixels_y

len_l_i1 = len(l_distances_all[0])
voids_looked_at = len_l_i1

In [ ]:
progress_bar(0, voids_looked_at-1)
for i0 in range(voids_looked_at):
    progress_bar(i0, voids_looked_at-1)

    densities_nonscaledx_nonscaledy.append([])
    densities_scaledx_scaledy.append(      [])
    
    densities = l_densities[i0].copy().tolist()
    

    # for the (min_grid, mean grid and ellipsoid) centers
    for i3 in range(3):
        
        # add to each grid distance all densities
        distances     = l_distances_all[  i3][i0].copy()
        all_distances = all_distances_all[i3]
        density_grid_scaled = {x: [] for x in all_distances}
        for dns, dist in zip(densities, distances): density_grid_scaled[round(dist,roundingterm)].append(dns)
        
        # take the mean or fill with -100 the empty distance bins
        density_grid_scaled     = [np.mean(_) if _      else -100  for _ in density_grid_scaled.values()]
        #min_density_grid_scaled = np.min([ _  if _>-100 else 10**7 for _ in density_grid_scaled])           # when picking the min, just use 100 for those -100 entries, so we know they won't count as the minimum
        #density_grid_scaled     = [_-min_density_grid_scaled       for _ in density_grid_scaled]
        density_grid_scaled     = [np.log10(np.where(-100 < _ < 10**min_density, 10**min_density, _)) for _ in density_grid_scaled]

        # interpolate the non-empty entries (so the same all_distances, just fill in the empty entries)
        valid_indices = np.where(np.array(density_grid_scaled) >= min_density)[0]
        density_grid_scaled = np.interp(all_distances, np.array(all_distances)[valid_indices], np.array(density_grid_scaled)[valid_indices]).tolist()

        # set everything after the max radius of this void to np.nan 
        first_nonvalid = valid_indices[-1]+1
        density_grid_scaled[first_nonvalid:] = [np.nan] * (len(all_distances) - first_nonvalid)

        # get the mean between fixed bins of size dx (unless the bin contains just one value, then the position is the same)
        x_new, y_new = create_new_lists0(all_distances, density_grid_scaled, dx)


        
        # interpolate in the max radius of fft/ellipsoid fine 10 ranges
        no_bins_nonscaledx = no_bins_nonscaledx_all[i3]
        bins_nonscaledx    = bins_nonscaledx_all[   i3]
        
        y_new1 = np.interp(bins_nonscaledx, x_new, y_new).tolist()
        densities_nonscaledx_nonscaledy[-1].append(y_new1)



        # interpolate and normalize
        # if bins_scaledx == bins_nonscaledx (as it is here), the interpolation gives the same result
        no_bins_scaledx = no_bins_scaledx_all[i3]
        bins_scaledx    = bins_scaledx_all[i3]
        
        valid_indices = np.array(y_new1) >= min_density   # to get rid of np.nan (that only occur after the max radius, since we have interpolated
                                                          #    values before that)... we already don't have any actual value below min_density
        max_bins_nonscaledx    = np.array(bins_nonscaledx)[valid_indices][-1]
        bins_nonscaledx_scaled = np.array(bins_nonscaledx)[valid_indices]/max_bins_nonscaledx*no_bins_scaledx
        
        y_new2  = np.interp(bins_scaledx, bins_nonscaledx_scaled, np.array(y_new1)[valid_indices]).tolist()
        y_new2 -= np.min(y_new2)
        y_new2 /= np.max(y_new2)
        densities_scaledx_scaledy[-1].append(y_new2)


        # pixel map for the ellipsoid origin case
        if i3 != 0:
            # we scale the density profile
            y_new3 = np.array(y_new2)*pixel_scalefactor
            
            for j0 in range(1, len(y_new3)):
                
                if np.isfinite(y_new3[j0-1]) and np.isfinite(y_new3[j0]):
                    
                    x0 = j0-1
                    x1 = j0
                    
                    y0 = round(y_new3[j0-1])
                    y1 = round(y_new3[j0])
                    
                    if y0 <= no_pixels_y and y1 <= no_pixels_y and y0 >= 0 and y1 >= 0:
                        
                        n_steps = abs(y1-y0)+1
                        
                        xs = np.round(np.linspace(x0, x1, n_steps)).astype(int)
                        ys = np.round(np.linspace(y0, y1, n_steps)).astype(int)
                        
                        for x, y in zip(xs, ys):
                            if y <= no_pixels_y and y >= 0: pixels_scaledx_scaledy[i3-1][y][x] += 1

In [ ]:
max_density_nonscaledy = [np.max([np.nanmax(np.array(_[i3])) for _ in densities_nonscaledx_nonscaledy]) for i3 in [1,2]]
pixel_scalefactor_nonscaledy = [no_pixels_y/(max_density_nonscaledy[i3-1]-min_density) for i3 in [1,2]]


for i3 in range(1,3):
    progress_bar(0, voids_looked_at-1)
    for i0 in range(voids_looked_at):
        progress_bar(i0, voids_looked_at-1)
    
        # ellipsoid origin case
        y_new3 = (np.array(densities_nonscaledx_nonscaledy[i0][i3])-min_density)*pixel_scalefactor_nonscaledy[i3-1]
        
        for j0 in range(1, len(y_new3)):
            
            if np.isfinite(y_new3[j0-1]) and np.isfinite(y_new3[j0]):
                
                x0 = j0-1
                x1 = j0
                
                y0 = round(y_new3[j0-1])
                y1 = round(y_new3[j0])
                
                if y0 <= no_pixels_y and y1 <= no_pixels_y and y0 >= 0 and y1 >= 0:
                    
                    n_steps = abs(y1-y0)+1
                    
                    xs = np.round(np.linspace(x0, x1, n_steps)).astype(int)
                    ys = np.round(np.linspace(y0, y1, n_steps)).astype(int)
                    
                    for x, y in zip(xs, ys):
                        if y <= no_pixels_y and y >= 0: pixels_nonscaledx_nonscaledy[i3-1][y][x] += 1

In [ ]:
with open(file_path_ellips_Z+"densities_nonscaledx_nonscaledy.pk", 'wb') as f: pkl.dump(densities_nonscaledx_nonscaledy, f)
with open(file_path_ellips_Z+"densities_scaledx_scaledy.pk",       'wb') as f: pkl.dump(densities_scaledx_scaledy,       f)
with open(file_path_ellips_Z+"pixels_scaledx_scaledy.pk",          'wb') as f: pkl.dump(pixels_scaledx_scaledy,          f)
with open(file_path_ellips_Z+"pixels_nonscaledx_nonscaledy.pk",    'wb') as f: pkl.dump(pixels_nonscaledx_nonscaledy,    f)
with open(file_path_ellips_Z+"max_density_nonscaledy.pk",          'wb') as f: pkl.dump(max_density_nonscaledy,          f)

---

In [ ]:
all_ddd0 = 0; all_ddd1 = 0; all_ddd2 = 0
for ddd0, ddd in enumerate(densities_nonscaledx_nonscaledy):
    if ddd[1][1] <  ddd[1][0]: all_ddd0 += 1
    if ddd[1][1] == ddd[1][0]: all_ddd1 += 1

In [ ]:
print("Out of a total of "+
      str(voids_looked_at)+
      " voids we looked at, "+
      str(all_ddd0)+
      " have a negative density gradient, which amounts for "+
      str(round(all_ddd0/voids_looked_at*100,2))+
      "% of the total... and " +
      str(all_ddd1)+
      " have a null density gradient, which amounts for "+
      str(round(all_ddd1/voids_looked_at*100,2))+
      "% of the total.")

---

In [ ]:
all_ddd0 = 0; all_ddd1 = 0; all_ddd2 = 0
for ddd0, ddd in enumerate(densities_nonscaledx_nonscaledy):
    if ddd[2][1] <  ddd[2][0]: all_ddd0 += 1
    if ddd[2][1] == ddd[2][0]: all_ddd1 += 1

In [ ]:
print("Out of a total of "+
      str(voids_looked_at)+
      " voids we looked at, "+
      str(all_ddd0)+
      " have a negative density gradient, which amounts for "+
      str(round(all_ddd0/voids_looked_at*100,2))+
      "% of the total... and " +
      str(all_ddd1)+
      " have a null density gradient, which amounts for "+
      str(round(all_ddd1/voids_looked_at*100,2))+
      "% of the total.")

---

In [ ]:
voids_printed = [0,4,10,17]

In [ ]:
origin_shift_frac = 0.055  # visual width of grey origin-shift region; tune this only

def apply_equal_origin_shift(ax, lines, scatters, frac=0.055, label=False):
    
    ax.relim(); ax.autoscale_view()

    x_right = ax.get_xlim()[1]
    shift = frac * x_right / (1.0 - frac)

    for line in lines:
        x = np.asarray(line.get_xdata(), dtype=float).copy()
        x[0] = -shift
        line.set_xdata(x)

    for sc in scatters:
        offsets = sc.get_offsets().copy()
        offsets[0, 0] = -shift
        sc.set_offsets(offsets)
        
    span = ax.axvspan(-shift, 0, color="grey", alpha=0.2, label="Origin shift" if label else None, zorder=0)
    ax.set_xlim(-shift, x_right)

    return shift, x_right, span


def nice_tick_label(x):
    if np.isclose(x, round(x)):
        return int(round(x))
    return latex_float(x)

In [ ]:
fig, axs = plt.subplots(2, 2, dpi=400, figsize=(12, 10))
axs = axs.flatten()

fig.suptitle(r"(Interpolated) density profile from baseline inside "+str(len(voids_printed))+" random voids as a function of their radius" + "\n"
             + "with their origin further shifted to the left for clarity" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]", y=0.89)

axs[0].set_title("Void average center")
axs[1].set_title("Ellipsoid center")


top_line_handles    = []; top_scatter_handles    = []
bottom_line_handles = []; bottom_scatter_handles = []



for i0, i00 in enumerate([1, 2]):

    bins_nonscaledx = np.asarray(bins_nonscaledx_all[i00], dtype=float).copy()
    bins_nonscaledxx = bins_nonscaledx.copy()
    bins_nonscaledxx[0] = 0.0

    lines = []; scatters = []
    for i1 in voids_printed:

        yy = densities_nonscaledx_nonscaledy[i1][i00].copy()
        yy = np.asarray(yy, dtype=float)
        yy[yy <= -5] = -5

        line, = axs[i0].plot(   bins_nonscaledxx, yy, lw=0.4, zorder=5, label=str(i1))
        scat  = axs[i0].scatter(bins_nonscaledxx, yy, s=0.3,  zorder=5)
        lines.append(line); scatters.append(scat)

    axs[i0].grid()

    top_line_handles.append(lines)
    top_scatter_handles.append(scatters)



for i0, i00 in enumerate([1, 2]):

    bins_scaledx = np.asarray(bins_scaledx_all[i00], dtype=float).copy()
    bins_scaledxx = bins_scaledx.copy()
    bins_scaledxx[0] = 0.0

    lines = []; scatters = []
    for i1 in voids_printed:

        yy = densities_scaledx_scaledy[i1][i00].copy()
        yy = np.asarray(yy, dtype=float)
        yy[yy <= -5] = -5

        line, = axs[i0 + 2].plot(   bins_scaledxx, yy, lw=0.4, zorder=5)
        scat  = axs[i0 + 2].scatter(bins_scaledxx, yy, s=0.3,  zorder=5)
        lines.append(line); scatters.append(scat)

    axs[i0 + 2].grid()

    bottom_line_handles.append(lines)
    bottom_scatter_handles.append(scatters)


top_shift_info = []; bottom_shift_info = []
for i0 in range(2):
    shift, x_right, _ = apply_equal_origin_shift(axs[i0], top_line_handles[i0], top_scatter_handles[i0], frac=origin_shift_frac, label=(i0 == 0))
    top_shift_info.append((shift, x_right))
    shift, x_right, _ = apply_equal_origin_shift(axs[i0 + 2], bottom_line_handles[i0], bottom_scatter_handles[i0], frac=origin_shift_frac, label=False)
    bottom_shift_info.append((shift, x_right))


for i0, i00 in enumerate([1, 2]):

    shift, x_right = top_shift_info[i0]
    xticks_auto = axs[i0].get_xticks()
    xticks_auto = [x for x in xticks_auto if (x > 0) and (x <= x_right)]

    xticks_top = [-shift] + xticks_auto
    xticklabels_top = [0] + [nice_tick_label(x) for x in xticks_auto]

    axs[i0].set_xticks(xticks_top, xticklabels_top)

    shift_scaled, x_right_scaled = bottom_shift_info[i0]
    xmax_scaled = np.nanmax(np.asarray(bins_scaledx_all[i00], dtype=float)[1:])
    xticks_bottom = [-shift_scaled] + [xmax_scaled / 10 * k for k in range(1, 11)]
    xticklabels_bottom = [0] + [10 * k for k in range(1, 11)]

    axs[i0 + 2].set_xticks(xticks_bottom, xticklabels_bottom)
    axs[i0 + 2].set_yticks([_ / 10 for _ in range(11)], [_ * 10 for _ in range(11)])
 
    max_val_i0 = np.max([np.max(np.asarray(oo1[i00])[np.asarray(oo1[i00]) >= -100]) for oo1 in [densities_nonscaledx_nonscaledy[o9] for o9 in voids_printed]])

    ytks, ytkss = set_ticks(10**min_density, 10**max_val_i0)

    ytks = [np.log10(_) for _ in ytks]
    axs[i0].set_yticks(ytks, ytkss)



axs[0].set_ylabel(r"$\delta$", labelpad=-15)
axs[2].set_ylabel(r"$\delta$ [%]")

axs[0].set_xlabel(r"Radius distance [cMpc/$h$]")
axs[1].set_xlabel(r"Radius distance [cMpc/$h$]")
axs[2].set_xlabel(r"Radius percentage distance [%]")
axs[3].set_xlabel(r"Radius percentage distance [%]")


handles, labels = axs[0].get_legend_handles_labels()

if "Origin shift" in labels:
    idx = labels.index("Origin shift")
    order = [idx] + [i for i in range(len(labels)) if i != idx]
    handles = [handles[i] for i in order]
    labels = [labels[i] for i in order]

axs[0].legend(handles, labels, loc=2)


plt.tight_layout(rect=[0, 0, 1, 0.90], pad=1.0, h_pad=3.0)

plt.savefig(plots_path_0 + "13.1___A_few_profiles___mindensity_and_ellipsoid_"+sel+"_origins.png", bbox_inches="tight", pad_inches=0.1)
plt.close()

---
---
---

## 13.2 Void profiles: all the selected

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod

---

In [ ]:
#sel = "" # ""

In [ ]:
file_path_ellips_Z = "../Analysis_Ellipsoid_Data/D___["+str(ud)+"_"+str(od)+"]/Z___"+Z+"/"
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_densities.pk"    ,    "rb") as f: l_densities     = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_densities_all.pk",    "rb") as f: l_densities_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_distances_all.pk",    "rb") as f: l_distances_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_radii_max_all.pk",    "rb") as f: l_radii_max_all = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_radii.pk"        ,    "rb") as f: l_radii         = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_coverage.pk"     ,    "rb") as f: l_coverage      = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_purity.pk"       ,    "rb") as f: l_purity        = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_IoU.pk"          ,    "rb") as f: l_IoU           = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_Dice.pk"         ,    "rb") as f: l_Dice          = pkl.load(f)
with open(file_path_ellips_Z+"filtered_vals/l"+sel+"_radii_sss.pk"    ,    "rb") as f: l_radii_sss     = pkl.load(f)

---

In [ ]:
with open(file_path_ellips_Z+"bins_nonscaledx_all.pk",    'rb') as f: bins_nonscaledx_all    = pkl.load(f)
with open(file_path_ellips_Z+"bins_scaledx_all.pk",       'rb') as f: bins_scaledx_all       = pkl.load(f)
with open(file_path_ellips_Z+"no_bins_nonscaledx_all.pk", 'rb') as f: no_bins_nonscaledx_all = pkl.load(f)
no_bins_scaledx_all = no_bins_nonscaledx_all

In [ ]:
# make sure they match
no_pixels_y = 4000
min_density = 0
dx = 0.2

In [ ]:
with open(file_path_ellips_Z+"pixels_scaledx_scaledy.pk",       'rb') as f: pixels_scaledx_scaledy       = pkl.load(f)
with open(file_path_ellips_Z+"pixels_nonscaledx_nonscaledy.pk", 'rb') as f: pixels_nonscaledx_nonscaledy = pkl.load(f)
with open(file_path_ellips_Z+"max_density_nonscaledy.pk",       'rb') as f: max_density_nonscaledy       = pkl.load(f)

pixel_scalefactor_nonscaledy = [no_pixels_y/(max_density_nonscaledy[i3-1]-min_density) for i3 in range(1,3)]

---

In [ ]:
pixel_scalefactor = no_pixels_y

len_l_i1 = len(l_distances_all[0])
voids_looked_at = len_l_i1

---

In [ ]:
def smooth_non_cyclic(arr, sigma, pad_factor=2):

    if sigma == 0: return arr
    
    ny, nx = arr.shape
    
    pad_ny = int(ny * (pad_factor - 1) // 2)
    pad_nx = int(nx * (pad_factor - 1) // 2)
    padded_arr = np.pad(arr, ((pad_ny, pad_ny), (pad_nx, pad_nx)), mode='reflect')
    
    fft_arr = fftn(padded_arr)
    smoothed_fft = fourier_gaussian(fft_arr, sigma=sigma)
    smoothed_padded = ifftn(smoothed_fft).real
    
    return smoothed_padded[pad_ny:pad_ny+ny, pad_nx:pad_nx+nx]

---

In [ ]:
fig, axs = plt.subplots(2, 2, dpi=400, figsize=(12,8))

fig.suptitle(r"(Interpolated) density profiles from baseline inside all ("+str(int(voids_looked_at))+") voids as a function of radius"
             +"\n"+r"with a no Gaussian filter applied" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


axs[0,0].set_title("Grid mean center: non-scaled x & y", fontsize=10)
axs[0,1].set_title("Ellipsoid center: scaled x & y",     fontsize=10)

images = [None, None]
for i1 in range(2):
    indx2 = i1+1
    
    nonzero_x_nonscaled = np.where(np.sum(pixels_nonscaledx_nonscaledy[i1], axis=0) > 0)[0]; x_max_nonscaled_pix = nonzero_x_nonscaled[-1]
    nonzero_y_nonscaled = np.where(np.sum(pixels_nonscaledx_nonscaledy[i1], axis=1) > 0)[0]; y_max_nonscaled_pix = nonzero_y_nonscaled[-1]
    
    x_max_nonscaled_val = bins_nonscaledx_all[indx2][x_max_nonscaled_pix]
    y_max_nonscaled_val = min_density + y_max_nonscaled_pix/pixel_scalefactor_nonscaledy[i1]
    
    for i0 in range(2):
        if i0 == 0:
            pscs_i0 = pixels_nonscaledx_nonscaledy[i1].copy().astype(float)
            no_bins_scaledy = no_pixels_y
            no_bins_x       = len(bins_nonscaledx_all[indx2])-1
        else:
            pscs_i0 = pixels_scaledx_scaledy[i1].copy().astype(float)
            no_bins_scaledy = no_pixels_y
            no_bins_x       = no_bins_scaledx_all[indx2]*10

        pscs_i0_fft = smooth_non_cyclic(pscs_i0, sigma=0)
        
        # mask empty cells instead of plotting them
        pscs_i0_fft_masked = np.ma.masked_where(pscs_i0_fft <= 0, pscs_i0_fft)
        
        cmap = plt.cm.inferno.copy()
        cmap.set_bad(alpha=0)   # masked cells become transparent
        
        images[i0] = axs[i0,i1].imshow(pscs_i0_fft_masked, norm=LogNorm(vmin=1, vmax=np.max(pscs_i0_fft)), cmap=cmap, aspect="auto", origin="lower")
        
        

        x_pix = np.arange(pscs_i0_fft.shape[1]); y_pix = np.arange(pscs_i0_fft.shape[0])
        
        col_sum = np.sum(pscs_i0_fft, axis=0)
        
        y_mean_pix = np.full(pscs_i0_fft.shape[1], np.nan)
        valid_cols = col_sum > 0
        
        y_mean_pix[valid_cols] = np.sum(pscs_i0_fft[:, valid_cols] * y_pix[:, None], axis=0) / col_sum[valid_cols]
        
        axs[i0,i1].plot(x_pix, y_mean_pix, lw=0.8, zorder=5, label="Mean profile")

        

        if i0 == 0:
            
            scale_fcty = pixel_scalefactor_nonscaledy[i1]
            
            axs[i0,i1].set_xlim(-0.5, x_max_nonscaled_pix*1.01)
            axs[i0,i1].set_ylim(-0.5, y_max_nonscaled_pix*1.01)
            
            tksx, tkssx = set_ticks(0, x_max_nonscaled_val, log_lin=False, int_if_possible=True, float_g=True, number_of_tks_clean_min=5)
            x_tick_pix = np.interp(np.array(tksx), bins_nonscaledx_all[1], np.arange(len(bins_nonscaledx_all[1])))
        
            ytks, ytkss = set_ticks(10**min_density, 10**y_max_nonscaled_val)
            ytks = [np.log10(_) for _ in ytks]
            
            axs[i0,i1].set_xticks(x_tick_pix, tkssx)
            axs[i0,i1].set_yticks((np.array(ytks)-min_density)*scale_fcty, ytkss)

        
        else:
            
            no_bins_scaledx = no_bins_scaledx_all[indx2]*10
            
            scale_fcty = no_bins_scaledy/100
            scale_fctx = no_bins_scaledx/100
            
            axs[i0,i1].set_xlim(-0.5, no_bins_scaledx+0.5)
            axs[i0,i1].set_ylim(-0.5, no_bins_scaledy+0.5)
            
            tks, tkss = set_ticks(0, 100, log_lin=False, int_if_possible=True, float_g=True, number_of_tks_clean_min=5)
            axs[i0,i1].set_xticks(np.array(tks)*scale_fctx, tkss)
            axs[i0,i1].set_yticks(np.array(tks)*scale_fcty, tkss)
        
        axs[i1,i0].grid()
    

for i1 in range(2):
    axs[0,i1].set_xlabel(r"Radius distance [cMpc/$h$]")
    axs[1,i1].set_xlabel(r"Radius distance (as a percent)")
for i1 in range(1):
    axs[0,i1].set_ylabel(r"$\delta$ (logarithmic scale)")
    axs[1,i1].set_ylabel(r"$\delta$ (percentage of logarithmic scale) [%]")

axs[0,0].legend()
plt.tight_layout()
plt.savefig(plots_path_0+"13.2___pixels___all_voids_nonscaled_and_scaled_"+sel+"___no_gaussian.png")
plt.close()

---

In [ ]:
gaussian_sigma = 20

In [ ]:
fig, axs = plt.subplots(2, 2, dpi=400, figsize=(12,8))

fig.suptitle(r"(Interpolated) density profiles from baseline inside all ("+str(int(voids_looked_at))+") voids as a function of radius"
             +"\n"+r"with a Gaussian filter applied: $\sigma=$"+str(gaussian_sigma)+" pixels" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


axs[0,0].set_title("Grid mean center: non-scaled x & y", fontsize=10)
axs[0,1].set_title("Ellipsoid center: scaled x & y",     fontsize=10)

images = [None, None]
for i1 in range(2):
    indx2 = i1+1
    
    nonzero_x_nonscaled = np.where(np.sum(pixels_nonscaledx_nonscaledy[i1], axis=0) > 0)[0]; x_max_nonscaled_pix = nonzero_x_nonscaled[-1]
    nonzero_y_nonscaled = np.where(np.sum(pixels_nonscaledx_nonscaledy[i1], axis=1) > 0)[0]; y_max_nonscaled_pix = nonzero_y_nonscaled[-1]
    
    x_max_nonscaled_val = bins_nonscaledx_all[indx2][x_max_nonscaled_pix]
    y_max_nonscaled_val = min_density + y_max_nonscaled_pix/pixel_scalefactor_nonscaledy[i1]
    
    for i0 in range(2):
        if i0 == 0:
            pscs_i0 = pixels_nonscaledx_nonscaledy[i1].copy().astype(float)
            no_bins_scaledy = no_pixels_y
            no_bins_x       = len(bins_nonscaledx_all[indx2])-1
        else:
            pscs_i0 = pixels_scaledx_scaledy[i1].copy().astype(float)
            no_bins_scaledy = no_pixels_y
            no_bins_x       = no_bins_scaledx_all[indx2]*10


        

        pscs_i0_fft = smooth_non_cyclic(pscs_i0, sigma=gaussian_sigma)
        pscs_i0_fft[pscs_i0_fft < 0.01] = 0
        
        # mask empty cells instead of plotting them
        pscs_i0_fft_masked = np.ma.masked_where(pscs_i0_fft <= 0, pscs_i0_fft)
        
        cmap = plt.cm.inferno.copy()
        cmap.set_bad(alpha=0)   # masked cells become transparent
        
        images[i0] = axs[i0,i1].imshow(pscs_i0_fft_masked, norm=LogNorm(vmin=0.01, vmax=np.max(pscs_i0_fft)), cmap=cmap, aspect="auto", origin="lower")
        
        

        x_pix = np.arange(pscs_i0_fft.shape[1]); y_pix = np.arange(pscs_i0_fft.shape[0])
        
        col_sum = np.sum(pscs_i0_fft, axis=0)
        
        y_mean_pix = np.full(pscs_i0_fft.shape[1], np.nan)
        valid_cols = col_sum > 0
        
        y_mean_pix[valid_cols] = np.sum(pscs_i0_fft[:, valid_cols] * y_pix[:, None], axis=0) / col_sum[valid_cols]
        
        axs[i0,i1].plot(x_pix, y_mean_pix, lw=0.8, zorder=5, label="Mean profile")

        

        if i0 == 0:
            
            scale_fcty = pixel_scalefactor_nonscaledy[i1]
            
            axs[i0,i1].set_xlim(-0.5, x_max_nonscaled_pix*1.01)
            axs[i0,i1].set_ylim(-0.5, y_max_nonscaled_pix*1.01)
            
            tksx, tkssx = set_ticks(0, x_max_nonscaled_val, log_lin=False, int_if_possible=True, float_g=True, number_of_tks_clean_min=5)
            x_tick_pix = np.interp(np.array(tksx), bins_nonscaledx_all[1], np.arange(len(bins_nonscaledx_all[1])))
        
            ytks, ytkss = set_ticks(10**min_density, 10**y_max_nonscaled_val)
            ytks = [np.log10(_) for _ in ytks]
            
            axs[i0,i1].set_xticks(x_tick_pix, tkssx)
            axs[i0,i1].set_yticks((np.array(ytks)-min_density)*scale_fcty, ytkss)

        
        else:
            
            no_bins_scaledx = no_bins_scaledx_all[indx2]*10
            
            scale_fcty = no_bins_scaledy/100
            scale_fctx = no_bins_scaledx/100
            
            axs[i0,i1].set_xlim(-0.5, no_bins_scaledx+0.5)
            axs[i0,i1].set_ylim(-0.5, no_bins_scaledy+0.5)
            
            tks, tkss = set_ticks(0, 100, log_lin=False, int_if_possible=True, float_g=True, number_of_tks_clean_min=5)
            axs[i0,i1].set_xticks(np.array(tks)*scale_fctx, tkss)
            axs[i0,i1].set_yticks(np.array(tks)*scale_fcty, tkss)
        
        axs[i0,i1].grid()
    

for i1 in range(2):
    axs[0,i1].set_xlabel(r"Radius distance [cMpc/$h$]")
    axs[1,i1].set_xlabel(r"Radius distance (as a percent)")
for i1 in range(1):
    axs[0,i1].set_ylabel(r"$\delta$ (logarithmic scale)")
    axs[1,i1].set_ylabel(r"$\delta$ (percentage of logarithmic scale) [%]")

axs[0,0].legend()
plt.tight_layout()
plt.savefig(plots_path_0+"13.2___pixels___all_voids_nonscaled_and_scaled_"+sel+"___gaussian_sigma_"+str(gaussian_sigma)+".png")
plt.close()

---
---
---

## 13.3 Void profiles: HSW fit

---

In [ ]:
x_pix      = [_/2090*100 for _ in x_pix]
y_mean_pix = [_/4000*100 for _ in y_mean_pix]

---

In [ ]:
x_data = copy.deepcopy(x_pix)
y_data = copy.deepcopy(y_mean_pix)

As an empirical reference for the measured void-density profiles, I compare the spherically averaged profiles to the Hamaus-Sutter-Wandelt (HSW) profile.
This profile is not an exact analytic prediction for every individual watershed void, since WVF basins can be aspherical, irregular and affected by substructure; it is instead used here as a smooth fitting template for the average radial trend.

The density field is written in terms of the density contrast $\delta(r)$ and the corresponding normalized density $q(r)$ as
\begin{equation}
\delta(r)=\frac{\rho(r)-\bar{\rho}}{\bar{\rho}}=\frac{\rho(r)}{\bar{\rho}}-1
\quad \Rightarrow \quad
q(r)\equiv \delta(r)+1=\frac{\rho(r)}{\bar{\rho}} .
\end{equation}
Thus $q=1$ corresponds to the cosmic mean density, $q<1$ to an underdensity and $q>1$ to an overdensity.
For a void of effective radius $R_v$, I use the dimensionless radius $s=r/R_v$; since the measured profiles are plotted in radius percent $x$, this gives
\begin{equation}
s=\frac{r}{R_v}=\frac{x}{100}.
\end{equation}

The HSW density-contrast profile is
\begin{equation}
\delta_{\mathrm{HSW}}(s)=\delta_{\mathrm{cen}}\frac{1-(s/s_c)^\alpha}{1+s^\beta},
\end{equation}
where $\delta_{\mathrm{cen}}$ is the central density contrast, $s_c=r_s/R_v$ is the mean-density crossing radius, $\alpha$ controls the inner rise and $\beta$ controls the steepness of the outer wall.
Since the measured quantity is $q=\delta+1$, the fitted density profile is
\begin{equation}
q_{\mathrm{HSW}}(s)=1+\delta_{\mathrm{cen}}\frac{1-(s/s_c)^\alpha}{1+s^\beta}.
\end{equation}
At the centre and at the mean-density crossing radius this gives
\begin{equation}
q_{\mathrm{HSW}}(0)=1+\delta_{\mathrm{cen}}
\quad \Rightarrow \quad
\delta_{\mathrm{cen}}>-1,
\qquad
s=s_c \quad \Rightarrow \quad \delta_{\mathrm{HSW}}(s_c)=0 \quad \Rightarrow \quad q_{\mathrm{HSW}}(s_c)=1.
\end{equation}
Because the plotted density is clipped below a finite floor $q_{\mathrm{floor}}$, the practical fitting bound is not merely $\delta_{\mathrm{cen}}>-1$, but
\begin{equation}
q_{\mathrm{HSW}}(0)\geq q_{\mathrm{floor}}
\quad \Rightarrow \quad
1+\delta_{\mathrm{cen}}\geq q_{\mathrm{floor}}
\quad \Rightarrow \quad
\delta_{\mathrm{cen}}\geq q_{\mathrm{floor}}-1.
\end{equation}
For the value used here, $q_{\mathrm{floor}}=10^{-5}$, this means $\delta_{\mathrm{cen}}\geq -0.99999$.

The model is fitted in the same clipped logarithmic percentage scale as the measured profiles.
First the physical density variable is clipped to the plotting range,
\begin{equation}
q_{\mathrm{clip}}=\min\left[\max(q,q_{\mathrm{floor}}),q_{\mathrm{ceiling}}\right],
\end{equation}
and then converted to a logarithmic coordinate $L=\log_{10}(q_{\mathrm{clip}})$.
For a fixed global range $L_{\min}=\log_{10}(q_{\mathrm{floor}})$ and $L_{\max}=\log_{10}(q_{\mathrm{ceiling}})$, the plotted density percentage is
\begin{equation}
Y(q)=100\frac{L-L_{\min}}{L_{\max}-L_{\min}}
=
100\frac{\log_{10}(q_{\mathrm{clip}})-\log_{10}(q_{\mathrm{floor}})}
{\log_{10}(q_{\mathrm{ceiling}})-\log_{10}(q_{\mathrm{floor}})}.
\end{equation}
For $q_{\mathrm{floor}}=10^{-5}$ and $q_{\mathrm{ceiling}}=10^3$, this becomes
\begin{equation}
Y(q)=100\frac{\log_{10}(q_{\mathrm{clip}})+5}{8}
\quad \Rightarrow \quad
Y(q=1)=100\frac{0+5}{8}=62.5.
\end{equation}
Therefore, on this logarithmic scale, the cosmic mean density is located at $62.5\%$, not at $50\%$.

Combining the HSW profile with this logarithmic conversion gives the full fitted model
\begin{equation}
Y_{\mathrm{model}}(x;\delta_{\mathrm{cen}},s_c,\alpha,\beta)
=
100
\frac{
\log_{10}\left[q_{\mathrm{clip}}(x;\delta_{\mathrm{cen}},s_c,\alpha,\beta)\right]
-
\log_{10}(q_{\mathrm{floor}})
}{
\log_{10}(q_{\mathrm{ceiling}})
-
\log_{10}(q_{\mathrm{floor}})
},
\end{equation}
with
\begin{equation}
q_{\mathrm{clip}}(x;\delta_{\mathrm{cen}},s_c,\alpha,\beta)
=
\min\left[
\max\left(
1+\delta_{\mathrm{cen}}
\frac{1-\left[(x/100)/s_c\right]^\alpha}{1+(x/100)^\beta},
q_{\mathrm{floor}}
\right),
q_{\mathrm{ceiling}}
\right].
\end{equation}
This is the function fitted to the measured profile when both the radius and density data are already expressed as percentages.

The residual at each sampled radius is defined as
\begin{equation}
\epsilon_i=y_i-Y_{\mathrm{model}}(x_i),
\end{equation}
where $y_i$ is the measured logarithmic density percentage and $Y_{\mathrm{model}}(x_i)$ is the fitted model value.
Thus $\epsilon_i>0$ means that the measured profile lies above the model, while $\epsilon_i<0$ means that it lies below it.
The reported fit statistics are
\begin{equation}
\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}\epsilon_i^2},
\qquad
\mathrm{MAE}=\frac{1}{N}\sum_{i=1}^{N}|\epsilon_i|,
\qquad
R^2=1-\frac{\sum_i\epsilon_i^2}{\sum_i(y_i-\bar{y})^2}.
\end{equation}
Since the fit is performed in percentage space, the RMSE and MAE are measured in percentage points.

This procedure is therefore a consistency test against a standard smooth empirical void-profile template.
It should not be interpreted as evidence that each individual WVF void must obey the HSW form exactly.
Systematic residuals, central bumps, oscillations or an unusually sharp wall can arise from real substructure, imperfect centring, ellipsoidal or irregular void geometry, interpolation effects, grid effects, or from the fact that the measured profile is not an ideal stacked spherical average.

In [ ]:
def hsw_q_from_percent_radius(x_percent, delta_c=-0.9999, s_c=0.85, alpha=2.0, beta=20.0):
    
    """
    Canonical Hamaus-Sutter-Wandelt profile in q = delta + 1 = rho/rhobar.

    x_percent : radius in percent, 0 to 100
    delta_c   : central density contrast
    s_c       : r_s / R_v, mean-density crossing radius
    alpha     : inner slope
    beta      : outer wall sharpness
    """
    
    s = np.asarray(x_percent, dtype=float) / 100.0

    delta = delta_c * (1.0 - (s / s_c)**alpha) / (1.0 + s**beta)
    q = 1.0 + delta

    return q

In [ ]:
def q_to_log_percent(q, q_floor=1e-5, q_ceiling=1e3):
    
    """
    Convert q = delta + 1 to a fixed clipped-log percentage scale.
    """
    
    q = np.asarray(q, dtype=float)

    q_clip = np.clip(q, q_floor, q_ceiling)

    L = np.log10(q_clip)
    Lmin = np.log10(q_floor)
    Lmax = np.log10(q_ceiling)

    return 100.0 * (L - Lmin) / (Lmax - Lmin)

In [ ]:
def hsw_y_percent_model(x_percent, delta_c, s_c, alpha, beta, q_floor=1e-5, q_ceiling=1e3):
    
    """
    Full model in the same units as y_data:
    radius percent -> HSW q -> clipped-log density percent.
    """
    
    q = hsw_q_from_percent_radius(x_percent, delta_c=delta_c, s_c=s_c, alpha=alpha, beta=beta)

    y_percent = q_to_log_percent(q, q_floor=q_floor, q_ceiling=q_ceiling)

    return y_percent

In [ ]:
def fit_hsw_percent_profile(x_data, y_data, q_floor=1e-5, q_ceiling=1e3, p0=None, title="HSW fit to percentage density profile"):
    
    """
    Fit the canonical HSW profile to x_data, y_data, where both are in percent.

    x_data : radius [%]
    y_data : clipped-log density scale [%]
    """

    x = np.asarray(x_data, dtype=float)
    y = np.asarray(y_data, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    mask &= (x >= 0.0) & (x <= 100.0)

    x_fit = x[mask]
    y_fit = y[mask]

    order = np.argsort(x_fit)
    x_fit = x_fit[order]
    y_fit = y_fit[order]

    delta_lower = q_floor - 1.0

    lower_bounds = [delta_lower, 0.15, 0.10, 0.10]
    upper_bounds = [-1e-12,     3.00, 30.0, 150.0]

    bounds = (lower_bounds, upper_bounds)

    if p0 is None: p0 = [-0.9999, 0.85, 2.0, 20.0]

    p0 = np.asarray(p0, dtype=float)

    eps = 1e-10
    p0 = np.maximum(p0, np.asarray(lower_bounds) + eps)
    p0 = np.minimum(p0, np.asarray(upper_bounds) - eps)

    def fit_func(xp, delta_c, s_c, alpha, beta):
        return hsw_y_percent_model(xp, delta_c, s_c, alpha, beta, q_floor=q_floor, q_ceiling=q_ceiling)

    popt, pcov = curve_fit(fit_func, x_fit, y_fit, p0=p0, bounds=bounds, maxfev=100000)

    y_best = fit_func(x_fit, *popt)
    residuals = y_fit - y_best

    rmse = np.sqrt(np.mean(residuals**2))
    mae = np.mean(np.abs(residuals))

    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_fit - np.mean(y_fit))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    print("Best-fit parameters:")
    print(f"delta_c = {popt[0]:.10g}")
    print(f"s_c     = {popt[1]:.10g}")
    print(f"alpha   = {popt[2]:.10g}")
    print(f"beta    = {popt[3]:.10g}")
    print()
    print("Fit quality in percentage points:")
    print(f"RMSE = {rmse:.6g}")
    print(f"MAE  = {mae:.6g}")
    print(f"R^2  = {r2:.6g}")
    print()
    print("Residual range:")
    print(f"min residual = {np.nanmin(residuals):.6g}")
    print(f"max residual = {np.nanmax(residuals):.6g}")


    x_model = np.linspace(0, 100, 1000)
    y_model = fit_func(x_model, *popt)

    plt.figure(figsize=(7.0, 4.4), dpi=180)
    plt.scatter(x_fit, y_fit, s=8, alpha=0.45, label="data")
    plt.plot(x_model, y_model, c="r", lw=2.2, label="HSW fit")
    plt.xlabel(r"Radius distance (as a percent)")
    plt.ylabel("log-density scale [%]")
    plt.xlim(0, 100)
    plt.ylim(-5, 105)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.grid()
    plt.savefig(plots_path_0+"13.2___HSW_fit_"+str(indx2)+".png")
    plt.close()

    plt.figure(figsize=(7.0, 3.0), dpi=180)
    plt.axhline(0, lw=1, c="black")
    plt.plot(x_fit, residuals, lw=1.5)
    plt.xlabel(r"Radius distance (as a percent)")
    plt.ylabel("data - model [% points]")
    plt.title("Residuals")

    lim = np.nanmax(np.abs(residuals))
    if np.isfinite(lim) and lim > 0:
        plt.ylim(-1.15 * lim, 1.15 * lim)

    plt.tight_layout()
    plt.grid()
    plt.savefig(plots_path_0+"13.2___HSW_residuals"+str(indx2)+".png")
    plt.close()

    return {"params": {"delta_c": popt[0], "s_c": popt[1], "alpha": popt[2], "beta": popt[3]},
            "popt": popt,
            "pcov": pcov,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "x_fit": x_fit,
            "y_fit": y_fit,
            "y_best": y_best,
            "residuals": residuals}

In [ ]:
result_hsw = fit_hsw_percent_profile(x_data, y_data, q_floor=1e-5, q_ceiling=1e3, p0=(-0.9999, 0.85, 2.0, 20.0), title="HSW fit to percentage density profile")

---
---
---